# CLDT-Thread — Phase 4: Frozen Shadow-Model Comparison
## Kalibrasi, Held-Out Load Step, dan Scoring Tiga Model

**Technical Source Baseline:** commit `0dee73247d4e87a3afec8fb90f37439cbb96523b`  
**Notebook Revision:** authored after re-auditing repository HEAD `2979cbb211ceba2cb9f914994dbc789147617694`. Perubahan setelah technical baseline hanya menyentuh notebook dan aset diagram; source C, header, schema, manifest, dan CMake yang disalin di sini tetap identik.

Phase 4 mengubah evidence Week 3 menjadi digital shadow yang dapat diuji. Hasil minimumnya bukan controller: tiga model dibekukan, mengeluarkan prediction pada horizon masa depan yang sama, lalu dinilai pada load step held-out tanpa satu pun remote policy.

| Area Lingkup | Sasaran dan Batasan Minimum |
|---|---|
| **Hasil Minimum** | Naive moving average, network-only, dan cross-layer model mempunyai frozen artifact identity, feature contract, horizon, metric-bound interval, calibration evidence, serta held-out score yang dibekukan. |
| **Jalur Aktif** | `host/kalman.*`, `host/twin_model.*`, `host/estimator.*`, shadow-only subset `host/coordinator.*`, Week-4 subset `host/analysis/reproduce.py`, dan pasangan `load-step.jsonc` / `load-step.json`. |
| **Batas Keras** | `remote_actuation_enabled == false`, `candidate_action == CLDT_ACTION_NONE`, dan tidak ada gate transition, policy publish, stale fallback, restart/replay, topology shift, ablation, SMP, power, dashboard, atau pembelian hardware baru. |
| **Exit yang Sah** | Positive, inconclusive, atau negative result sama-sama sah bila freeze, identity, raw evidence, reconciliation, dan uncertainty lengkap. Cross-layer yang gagal tidak diganti post hoc; actuation tetap mati. |

## Peta Kerja Week 4
### Dari Reconciled Baseline ke Frozen Held-Out Result

Diagram ini menempatkan contract repair sebelum model code. Jalur physical held-out hanya terbuka setelah feature set, horizon, calibration binding, numerical matrices, load-step semantics, dan verification boundary dapat diuji.

![Phase 4 implementation roadmap](docs/diagrams/phase4/implementation-roadmap.svg)

[Mermaid source](docs/diagrams/phase4/implementation-roadmap.mmd)

## Notebook Execution Boundary
### Scratch Copy, Bukan Repository Adapter

Cell `%%writefile` di notebook ini hanya membuat salinan source saat ini di `/content/cldt_scratch` agar TODO dapat dilihat dan diberi catatan tanpa mengubah checkout. Salinan itu bukan source of truth, tidak di-compile otomatis, tidak menulis evidence, dan tidak terhubung ke board. Implementasi tetap dilakukan pada file repository dengan nama path yang tertulis pada setiap heading.

Cell berikut hanya membuat direktori scratch yang memang dibutuhkan oleh `%%writefile`. Terminal CMake, ESP-IDF, broker, serial monitor, dan physical run tetap berjalan pada workstation/repository lokal.

In [ ]:
from pathlib import Path
Path("/content/cldt_scratch").mkdir(parents=True, exist_ok=True)


## Step 1: Entry Gate dari Phase 3
### Model Claim Dimulai dari Evidence yang Sudah Reconcile

Source model boleh dikerjakan lebih awal, tetapi physical held-out claim belum dimulai bila baseline end-to-end masih berupa pilot yang tidak dapat direplay. Entry record berikut disalin dari evidence Phase 3; kolom contoh hanya memperlihatkan format.

| Evidence masuk | Contoh format — diganti dengan hasil aktual |
|---|---|
| Source revision | e.g. 40 karakter hexadecimal |
| Ready baseline manifest dan digest | e.g. exact archived bytes + SHA-256 |
| Control profile identity | e.g. exact `profile_id`, `calibration_id`, dan resolved digest dari registry nyata |
| Binary identity gateway/RCP/A/B | e.g. empat SHA-256, masing-masing 64 hexadecimal |
| Stable physical block | e.g. channel, placement, role, parent, partition, dan power path tetap |
| Repeated baseline terminal status | e.g. seluruh planned run tercatat complete/invalid/interrupted |
| `cldt_item_audit_t.consistent` | e.g. true pada setiap run yang dipakai |
| Semua `cldt_reconciliation_t.consistent` | e.g. true untuk traffic class yang dinilai |
| Trace/drop accounting | e.g. source presence lengkap; drop count dan queue high-water tersimpan |
| Clock rule | e.g. mapped time valid dengan uncertainty, atau one-way feature dinonaktifkan |
| Raw-first proof | e.g. recorder append terjadi sebelum decode/model update |
| Lifecycle replay exit code | e.g. 0 pada evidence bundle yang dipakai |

Urutan masuknya:

1. Baseline yang failed reconciliation tidak menjadi calibration data. Raw bytes tetap disimpan dan run tetap tercatat invalid.
2. Missing source, stale observation, clock ambiguity, dan topology drift tidak diubah menjadi angka residual yang terlihat baik.
3. Source/model work dapat berlanjut pada fixed synthetic inputs ketika hardware gate belum hijau, tetapi istilah “held-out physical result” belum dipakai.
4. Perubahan board, binary, firmware, channel, placement, role/parent/partition, observation encoding, atau load-step scheduler memulai calibration block dan frozen model artifact baru. Load-step-capable firmware dibekukan sebelum calibration baseline, bukan disisipkan hanya pada held-out run.
5. Phase 4 tidak memperbaiki Phase 3 dengan menyisipkan model sebelum recorder. Evidence path tetap raw-first.

> **Cut rule:** tanpa baseline Phase 3 yang berulang dan reconcile, Phase 4 berhenti pada contract/numerical implementation. Load-step manifest tetap template dan tidak ada model claim.

## Step 2: Batas File dan Dependency
### Direct Work, Reopened Dependency, dan Deferred Work

| Kelompok | File yang benar-benar terkait | Perlakuan Phase 4 |
|---|---|---|
| Direct model | `host/kalman.h`, `host/kalman.c`, `host/twin_model.h`, `host/twin_model.c`, `host/estimator.h`, `host/estimator.c` | Header contract ditutup dahulu; badan TODO kemudian diimplementasikan. |
| Direct orchestration | `host/coordinator.h`, `host/coordinator.c`, `host/CMakeLists.txt` | Hanya branch shadow; pending horizon dan bounded ownership harus nyata. |
| Direct reproduction | `host/analysis/reproduce.py` | Lifecycle preflight Phase 3 dipertahankan; tiga-model fit/score dan run-aware uncertainty ditambahkan. |
| Direct experiment | `experiments/authoring/load-step.jsonc`, `experiments/load-step.json`, `schemas/experiment.schema.json` | JSONC diisi dari pilot; strict JSON baru ready setelah executable semantics tidak ambigu. |
| Reopened prerequisite | `cldt_types.h`, `cldt_metrics.h`, `cldt_control_profile.h`, `experiment_config.*`, `workload.*` | Bukan ditulis ulang; hanya contract yang dibutuhkan load step/horizon diperiksa dan, bila perlu, direvisi pada owner aslinya. |
| Phase 3 prerequisites | observation payload codec, MAC/queue/RTOS mapping, clock mapping, broker, recorder, lifecycle audit, aggregate reconciliation | Harus sudah bekerja. Jika masih stub, live shadow belum eligible. |
| Week 5 | `fidelity_gate.*`, `policy.*`, command/auth apply, `policy_guard_accept()`, stale fallback, restart/replay | Tidak dipanggil dan tidak “disiapkan sekalian”. |
| Conditional/deferred | ablation, topology shift, SMP, power, dashboard, sensors, node tambahan | Tidak masuk. |

Model host sendiri tidak memerlukan firmware image khusus. Namun executable local load-step scheduler belum ada pada contract Week 3; bila `workload.*`, runtime header, atau schema harus direvisi untuk menambahkannya, firmware final tersebut dibangun dan dibekukan sebelum calibration block. Calibration baseline (step disabled) dan held-out load-step (step enabled by admitted schedule) wajib memakai source, `sdkconfig`, dan binary yang sama. Evidence baseline lama dengan binary berbeda tetap berguna sebagai bring-up evidence, tetapi tidak menjadi calibration comparator untuk final Week-4 score.

## Step 3: Contract Repair Sebelum Mengisi Badan TODO
### Sebelas Keputusan yang Saat Ini Belum Dapat Dibuktikan dari Signature

Masalah berikut bukan tambahan scope. Masing-masing adalah informasi yang diminta komentar TODO atau dokumen metodologi, tetapi belum dapat direpresentasikan oleh header/runtime saat ini. Mengisi badan fungsi sebelum keputusan ini ditutup akan menghasilkan implementasi yang terlihat lengkap tetapi tidak dapat membuktikan comparison yang dimaksud.

| No. | Gap pada contract yang ada | Bentuk penyelesaian yang dibutuhkan sebelum code lanjut |
|---:|---|---|
| 1 | Observation wire payload belum mendefinisikan mapping `cldt_trace_record_t`, MAC diagnostic, dan host receive time | Satu encoder/decoder owner dari Phase 3, fixed vectors, source-presence rule, dan mapping `detail[24]` yang tidak ditafsir berbeda oleh gateway/host. |
| 2 | `cldt_twin_model_t` tidak menyimpan mapping arbitrary `node_id` ke slot, last `boot_id`, `sequence`, atau validity | Satu bounded mapping/order contract yang dapat menolak duplicate/old record tanpa memakai `node_id` sebagai array index. |
| 3 | `cldt_twin_model_observe()` hanya menerima device-local time, sedangkan model menggabungkan node | Satu host-time ownership rule; record tidak boleh memajukan `modeled_time_us` dengan clock domain yang tidak sebanding. |
| 4 | `calibrated` ada, tetapi tidak ada freeze transition; `cldt_estimator_observe()` dapat terus belajar | Satu explicit calibration-only → frozen transition. Held-out observation hanya menilai, tidak memperbarui parameter. |
| 5 | `cldt_prediction_t` dan `cldt_metrics_t` tidak membawa semua run/policy/horizon identity yang diminta scorer; satu pasangan interval juga belum terikat ke salah satu dari beberapa predicted metric atau unitnya | Satu observed-horizon object/owner yang mengikat exact run, epoch, start/end, completeness, dan reconciliation sebelum score, serta satu target/units identity pada prediction contract sebelum interval diinterpretasikan. |
| 6 | `cldt_metrics_t` bersifat cumulative, bukan metric untuk satu horizon | Snapshot-delta/aggregation rule yang mencegah warm-up, cooldown, atau horizon lain ikut ke target. |
| 7 | `cldt_kalman_t` tidak dimiliki model/estimator/coordinator; update menerima lima measurement sekaligus tanpa missing mask/time | Ownership, timestamp, initialization, partial-observation, dan singular-matrix failure rule dibekukan sebelum matrix code. |
| 8 | Naive window dan network/cross feature allowlist tidak ada di config/header; fixed 5-state design memuat cross-layer state | Naive state/window yang jelas dan matched observation mask untuk network/cross dengan model family/loss/horizon yang sama. |
| 9 | Coordinator tidak memiliki bounded inbound ownership atau pending prediction/horizon storage | Satu bounded owner untuk accepted observations dan satu bounded owner untuk prediction yang menunggu horizon selesai. |
| 10 | Manifest hanya memiliki satu `traffic.streams`; tidak ada structured before/during step atau baseline-evidence reference | Exact interpretation load step dan binding baseline/calibration harus masuk machine contract; jangan mengarang key karena schema memakai `additionalProperties: false`. |
| 11 | Tidak ada test target model/Kalman/estimator dan tidak ada Python dependency lock | Numerical, ordering, leakage, parity, and replay checks memerlukan owner nyata sebelum frozen result disebut reproducible. |

Notebook tidak memberi nama field baru untuk menyamarkan gap tersebut. Nama dan bentuk interface diputuskan di header/schema yang memang memiliki ownership. Catatan keputusan minimal:

| Human label | Bentuk isian |
|---|---|
| Observation payload owner | e.g. path source/header + fixed-vector test yang benar-benar dipilih |
| Node-to-slot and ordering owner | e.g. exact header yang direvisi + rejection cases |
| Model clock domain | e.g. host monotonic; mapping rule dan uncertainty bound |
| Calibration freeze boundary | e.g. artifact/revision yang membuat update berhenti |
| Horizon identity owner | e.g. existing/revised type dan exact fields |
| Kalman owner | e.g. model atau estimator; satu owner saja |
| Naive window | e.g. frozen duration/count dari calibration plan |
| Network feature allowlist | e.g. exact pre-issuance fields |
| Cross-layer feature allowlist | e.g. network fields + exact MAC/queue/RTOS fields |
| Load-step semantics | e.g. before state, during state, `at_s` reference, duration, acknowledgement |
| Deterministic verification owner | e.g. exact registered test target setelah repo change disetujui |

## Contract Vocabulary
### `common/include/cldt/cldt_types.h`

Header berikut adalah salinan repository. Dua enum dan satu fidelity record sudah ada, tetapi keberadaan type belum membuktikan model path bekerja. `cldt_fidelity_sample_t` boleh dihasilkan sebagai offline/shadow score pada Phase 4; ia belum dipakai untuk mengubah `cldt_gate_state_t`.

In [ ]:
%%writefile /content/cldt_scratch/cldt_types.h
#ifndef CLDT_TYPES_H
#define CLDT_TYPES_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#ifdef __cplusplus
extern "C" {
#endif

#define CLDT_PROTOCOL_MAGIC UINT16_C(0x434C)
#define CLDT_PROTOCOL_VERSION UINT8_C(1)
#define CLDT_WIRE_HEADER_BYTES UINT16_C(72)
#define CLDT_MAX_PAYLOAD_BYTES UINT16_C(256)
#define CLDT_AUTH_TAG_BYTES 16U
#define CLDT_TRACE_DETAIL_BYTES 24U
#define CLDT_POLICY_STREAM_COUNT 4U
#define CLDT_COMMAND_AUTHORITY_NODE_ID UINT32_C(0)

typedef uint32_t cldt_node_id_t;
typedef uint64_t cldt_run_id_t;
typedef uint32_t cldt_boot_id_t;
typedef uint32_t cldt_sequence_t;
typedef uint32_t cldt_policy_epoch_t;

typedef enum {
    CLDT_NODE_GATEWAY_HOST = 0,
    CLDT_NODE_RADIO_COPROCESSOR,
    CLDT_NODE_ROUTER_ENDPOINT,
    CLDT_NODE_LOW_POWER_ENDPOINT
} cldt_node_role_t;

typedef enum {
    CLDT_TRAFFIC_CONTROL = 0,
    CLDT_TRAFFIC_CRITICAL,
    CLDT_TRAFFIC_TELEMETRY,
    CLDT_TRAFFIC_BULK,
    CLDT_TRAFFIC_COUNT
} cldt_traffic_class_t;

typedef enum {
    CLDT_FRAME_OBSERVATION = 0,
    CLDT_FRAME_COMMAND,
    CLDT_FRAME_ACKNOWLEDGEMENT,
    CLDT_FRAME_CLOCK_SYNC,
    CLDT_FRAME_HEALTH
} cldt_frame_kind_t;

typedef enum {
    CLDT_EVENT_TASK_RELEASE = 0,
    CLDT_EVENT_TASK_START,
    CLDT_EVENT_TASK_FINISH,
    CLDT_EVENT_TASK_BLOCK,
    CLDT_EVENT_QUEUE_ENQUEUE,
    CLDT_EVENT_QUEUE_DEQUEUE,
    CLDT_EVENT_QUEUE_REJECT,
    CLDT_EVENT_POOL_EXHAUSTION,
    CLDT_EVENT_MESSAGE_SEND,
    CLDT_EVENT_MESSAGE_ACK,
    CLDT_EVENT_MESSAGE_EXPIRE,
    CLDT_EVENT_MESSAGE_COALESCE,
    CLDT_EVENT_MESSAGE_DROP,
    CLDT_EVENT_MESSAGE_DUPLICATE,
    CLDT_EVENT_LINK_CHANGE,
    CLDT_EVENT_POWER_SAMPLE,
    CLDT_EVENT_POLICY_APPLY,
    CLDT_EVENT_POLICY_REJECT,
    CLDT_EVENT_POLICY_FALLBACK,
    CLDT_EVENT_HEALTH,
    CLDT_EVENT_COUNT
} cldt_event_kind_t;

typedef enum {
    CLDT_GATE_COLD = 0,
    CLDT_GATE_OBSERVE,
    CLDT_GATE_TRUSTED,
    CLDT_GATE_ABSTAIN
} cldt_gate_state_t;

typedef enum {
    CLDT_MODEL_NAIVE = 0,
    CLDT_MODEL_NETWORK_ONLY,
    CLDT_MODEL_CROSS_LAYER,
    CLDT_MODEL_VARIANT_COUNT
} cldt_model_variant_t;

/*
 * In-memory metadata. It is not a packed wire structure. Encoding and decoding
 * must be performed field by field through cldt_protocol.h.
 *
 * Identity is frame-kind specific. For observations, acknowledgements, health,
 * and trace-bearing frames, node_id/boot_id identify the emitting device. A
 * version 1 command is one global policy datagram for every endpoint admitted
 * to the run: node_id is CLDT_COMMAND_AUTHORITY_NODE_ID and boot_id identifies
 * the host coordinator process that issued it, not a destination. The gateway
 * guards and forwards those identical bytes. Version 1 does not define
 * different authenticated command bytes per endpoint.
 */
typedef struct {
    cldt_frame_kind_t kind;
    cldt_traffic_class_t traffic_class;
    uint16_t flags;
    uint8_t hop_limit;
    cldt_node_id_t node_id;
    cldt_boot_id_t boot_id;
    cldt_sequence_t sequence;
    cldt_policy_epoch_t policy_epoch;
    cldt_run_id_t run_id;
    uint64_t transmit_local_us;
    uint64_t deadline_local_us;
} cldt_frame_meta_t;

/*
 * Decoder output borrows payload memory from the input byte buffer. The caller
 * must keep that buffer alive and unchanged while this view is in use.
 */
typedef struct {
    cldt_frame_meta_t meta;
    const uint8_t *payload;
    uint16_t payload_bytes;
    uint32_t crc32c;
    uint8_t authentication_tag[CLDT_AUTH_TAG_BYTES];
} cldt_frame_view_t;

typedef struct {
    cldt_event_kind_t kind;
    /* Every work-item event carries its class; HEALTH may use CLDT_TRAFFIC_COUNT. */
    cldt_traffic_class_t traffic_class;
    cldt_node_id_t node_id;
    cldt_boot_id_t boot_id;
    cldt_sequence_t sequence;
    cldt_policy_epoch_t policy_epoch;
    cldt_run_id_t run_id;
    uint64_t local_time_us;
    /*
     * Work-item events repeat the item's release and absolute deadline in the
     * same local monotonic clock domain as local_time_us. Non-work-item events
     * store zero in both fields. This permits stateless aggregate timing while
     * preserving raw timestamps for a separate per-item lifecycle audit.
     */
    uint64_t release_local_us;
    uint64_t deadline_local_us;
    uint32_t task_id;
    int8_t core_id;
    uint16_t queue_depth;
    int16_t link_rssi_dbm;
    uint32_t time_uncertainty_us;
    /* Fixed-size auxiliary bytes; each event kind documents its own encoding. */
    uint8_t detail[CLDT_TRACE_DETAIL_BYTES];
} cldt_trace_record_t;

typedef struct {
    uint32_t release_period_ms[CLDT_POLICY_STREAM_COUNT];
    uint32_t phase_offset_ms[CLDT_POLICY_STREAM_COUNT];
    uint16_t burst_limit[CLDT_POLICY_STREAM_COUNT];
    uint16_t batch_size[CLDT_POLICY_STREAM_COUNT];
    uint32_t token_rate_milli_pps[CLDT_POLICY_STREAM_COUNT];
    cldt_policy_epoch_t epoch;
    uint64_t issued_gateway_us;
    uint32_t ttl_ms;
} cldt_policy_t;

typedef struct {
    cldt_model_variant_t model_variant;
    uint64_t model_revision;
    uint64_t horizon_start_host_us;
    uint64_t horizon_end_host_us;
    uint64_t evaluated_host_us;
    uint64_t newest_observation_host_us;
    uint32_t sample_count;
    uint32_t model_lag_us;
    uint32_t clock_uncertainty_us;
    double relative_p95_error;
    double pdr_error_points;
    double prediction_interval_coverage;
    /* False when required horizon evidence is missing, stale, or unreconciled. */
    bool observation_integrity_valid;
    bool inside_calibrated_region;
} cldt_fidelity_sample_t;

#ifdef __cplusplus
}
#endif

#endif


### `common/include/cldt/cldt_metrics.h`

`cldt_metrics_t` merangkum satu measurement block secara cumulative. Scorer Phase 4 membutuhkan observed outcome untuk exact future horizon; karena header ini tidak memiliki run, epoch, atau horizon boundary, cumulative struct tidak boleh langsung diperlakukan sebagai horizon metric tanpa aggregation contract yang dibekukan.

In [ ]:
%%writefile /content/cldt_scratch/cldt_metrics.h
#ifndef CLDT_METRICS_H
#define CLDT_METRICS_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "cldt/cldt_status.h"
#include "cldt/cldt_types.h"

#ifdef __cplusplus
extern "C" {
#endif

/*
 * All counters refer to unique logical work items identified by run, node,
 * boot, and sequence. A transport retry is trace detail, not another sent
 * item. Aggregate counters are useful only after the item-identity audit below;
 * equal totals alone cannot prove that one item was not counted twice while
 * another item disappeared.
 */
typedef struct {
    uint64_t released[CLDT_TRAFFIC_COUNT];
    uint64_t admitted[CLDT_TRAFFIC_COUNT];
    uint64_t sent[CLDT_TRAFFIC_COUNT];
    uint64_t acknowledged[CLDT_TRAFFIC_COUNT];
    uint64_t on_time_acknowledged[CLDT_TRAFFIC_COUNT];
    uint64_t deadline_missed[CLDT_TRAFFIC_COUNT];
    uint64_t expired[CLDT_TRAFFIC_COUNT];
    uint64_t coalesced[CLDT_TRAFFIC_COUNT];
    uint64_t rejected[CLDT_TRAFFIC_COUNT];
    uint64_t dropped[CLDT_TRAFFIC_COUNT];
    uint64_t duplicated[CLDT_TRAFFIC_COUNT];
    uint64_t response_time_sum_us[CLDT_TRAFFIC_COUNT];
    uint64_t lateness_sum_us[CLDT_TRAFFIC_COUNT];
    uint32_t queue_high_water;
    uint32_t pool_exhaustions;
    uint64_t measurement_duration_us;
    uint64_t energy_uj;
} cldt_metrics_t;

/*
 * Reconciliation is a computed report, not a mutable metric. A class is
 * consistent only when every released work item is terminal or explicitly
 * unresolved at the snapshot boundary.
 */
typedef struct {
    uint64_t terminal[CLDT_TRAFFIC_COUNT];
    uint64_t unresolved[CLDT_TRAFFIC_COUNT];
    bool consistent[CLDT_TRAFFIC_COUNT];
} cldt_reconciliation_t;

/*
 * Result of auditing raw work-item lifecycles by full logical identity. This
 * is deliberately separate from aggregate reconciliation so reports cannot
 * mistake balanced counter corruption for complete evidence.
 */
typedef struct {
    uint64_t logical_items;
    uint64_t duplicate_releases;
    uint64_t duplicate_terminals;
    uint64_t terminal_without_release;
    uint64_t unresolved_items;
    bool consistent;
} cldt_item_audit_t;

void cldt_metrics_reset(cldt_metrics_t *metrics);

cldt_status_t cldt_metrics_record_trace(
    cldt_metrics_t *metrics,
    const cldt_trace_record_t *record);

/*
 * Calculates per-class aggregate conservation without modifying metrics. This
 * is a necessary check, not proof of per-item uniqueness. The caller must also
 * require a consistent cldt_item_audit_t and archive unresolved work rather
 * than discarding it before computing rates.
 */
cldt_status_t cldt_metrics_reconcile(
    const cldt_metrics_t *metrics,
    cldt_reconciliation_t *output);

/*
 * Audits work-item lifecycles in a trace sorted lexicographically by run_id,
 * node_id, boot_id, sequence, and local_time_us. The function skips non-item
 * events and never reorders caller-owned storage. Sorting belongs to the host
 * analysis/recorder boundary because embedded targets must not allocate an
 * unbounded identity table. A reportable run requires this audit and aggregate
 * reconciliation to pass.
 */
cldt_status_t cldt_metrics_audit_sorted_trace(
    const cldt_trace_record_t *records,
    size_t record_count,
    cldt_item_audit_t *output);

#ifdef __cplusplus
}
#endif

#endif


Pembacaan contract tersebut menghasilkan empat aturan implementasi:

1. Logical item tetap diidentifikasi oleh `run_id`, `node_id`, `boot_id`, dan `sequence`; transport retry bukan sample baru.
2. Horizon hanya eligible setelah item audit dan aggregate reconciliation untuk evidence yang menjadi target selesai.
3. Prediction issue time harus lebih awal atau sama dengan horizon start. Feature/outcome yang selesai setelah issue time tidak masuk predictor.
4. Missing, stale, contradictory, atau unreconciled horizon tetap muncul dengan `observation_integrity_valid == false`; row tidak dihapus dari denominator secara diam-diam.

| Contract check | Contoh format setelah implementasi |
|---|---|
| Same horizon untuk tiga variant | e.g. exact start/end identical |
| Warm-up di luar target | e.g. PASS |
| Cumulative-to-horizon conversion | e.g. fixed test dengan dua adjacent horizons |
| Future feature leakage | e.g. rejected |
| Missing source | e.g. retained as integrity failure |
| Duplicate score untuk satu horizon | e.g. rejected |

## Step 4: Calibration Identity dan Load-Step Admission
### Profile dan Executable Manifest Subset

`cldt_control_profile_t.calibration_id` adalah identity yang sudah tersedia untuk mengikat calibration evidence. Registry/profile document tetap dependency dari Phase 3; string yang tampak meyakinkan tidak menggantikan document dan digest.

In [ ]:
%%writefile /content/cldt_scratch/cldt_control_profile.h
#ifndef CLDT_CONTROL_PROFILE_H
#define CLDT_CONTROL_PROFILE_H

#include <stdint.h>

#include "cldt/cldt_status.h"
#include "cldt/cldt_types.h"

#ifdef __cplusplus
extern "C" {
#endif

/*
 * A ready manifest names one immutable control profile. The profile is resolved
 * by the host before a run starts, then its ID and digest are archived with the
 * evidence. It contains only safety-relevant selection values shared across
 * host and edge boundaries; it is not a generic configuration database.
 */
#define CLDT_CONTROL_PROFILE_ID_BYTES 65U
#define CLDT_CONTROL_PROFILE_DIGEST_BYTES 32U

typedef struct {
    /*
     * profile_id identifies the exact safety selection named by
     * treatment.control_profile in a ready manifest. calibration_id identifies
     * the separately versioned model-calibration evidence that supplies
     * residual and interval limits to the host fidelity gate.
     */
    char profile_id[CLDT_CONTROL_PROFILE_ID_BYTES];
    char calibration_id[CLDT_CONTROL_PROFILE_ID_BYTES];
    /* Version one permits actuation only from the frozen cross-layer candidate. */
    cldt_model_variant_t actuation_model_variant;

    /*
     * resolved_digest is the digest of the canonical, fully resolved profile
     * document. It prevents the same human-readable ID from silently referring
     * to different values in two evidence bundles. Digest calculation belongs
     * to the host registry/parser, not this portable validation function.
     */
    uint8_t resolved_digest[CLDT_CONTROL_PROFILE_DIGEST_BYTES];

    /* Host-side freshness and hysteresis inputs. */
    uint32_t maximum_observation_age_ms;
    uint16_t passing_windows_to_trust;

    /* Edge-side policy bounds. Compiled gateway/endpoints may be stricter. */
    uint32_t maximum_policy_ttl_ms;
    uint32_t maximum_total_rate_pps;
    uint32_t minimum_critical_period_ms;
    uint16_t maximum_bulk_burst_packets;
} cldt_control_profile_t;

/*
 * Validates only intrinsic profile shape and arithmetic safety. It performs no
 * file I/O, cryptographic digest calculation, model fitting, or device query.
 *
 * The later implementation must reject null/empty/non-terminated identifiers,
 * an all-zero digest, zero time/rate limits, and relationships that would make
 * a policy impossible to evaluate safely. It must leave caller-owned profile
 * bytes unchanged and return CLDT_ERR_NOT_IMPLEMENTED until those checks exist.
 */
cldt_status_t cldt_control_profile_validate(
    const cldt_control_profile_t *profile);

#ifdef __cplusplus
}
#endif

#endif


### `host/experiment_config.h`

Header ini menyediakan exact runtime subset untuk prediction treatment dan load-step scenario. Ia belum membawa baseline bundle, model matrices, feature labels, horizon schedule, atau before/during workload pair.

In [ ]:
%%writefile /content/cldt_scratch/experiment_config.h
#ifndef CLDT_HOST_EXPERIMENT_CONFIG_H
#define CLDT_HOST_EXPERIMENT_CONFIG_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "cldt/cldt_status.h"
#include "cldt/cldt_types.h"

#ifdef __cplusplus
extern "C" {
#endif

/*
 * This is the executable subset of a manifest, not a mirror of every planning
 * field in JSON. A state == "template" document is intentionally incomplete and
 * must never be converted into this structure. Parse only a completed
 * state == "ready" manifest after JSON Schema validation.
 */
#define CLDT_MAX_MANIFEST_NODES 4U
#define CLDT_MAX_WORKLOADS 4U
#define CLDT_CONFIG_DIGEST_BYTES 32U
#define CLDT_MANIFEST_ID_BYTES 65U
#define CLDT_MANIFEST_TEXT_BYTES 161U

typedef enum {
    CLDT_TREATMENT_BASELINE = 0,
    CLDT_TREATMENT_PREDICTION,
    CLDT_TREATMENT_GATED_CONTROL,
    CLDT_TREATMENT_SAFETY,
    CLDT_TREATMENT_SMP,
    CLDT_TREATMENT_POWER
} cldt_treatment_mode_t;

typedef enum {
    CLDT_ACTION_NONE = 0,
    CLDT_ACTION_BULK_RATE_REDUCE,
    /* Reserved for explicitly admitted future phases; rejected by v1 control. */
    CLDT_ACTION_PHASE_STAGGER,
    CLDT_ACTION_POWER_PROFILE
} cldt_candidate_action_t;

typedef enum {
    CLDT_SCENARIO_NONE = 0,
    CLDT_SCENARIO_LOAD_STEP,
    CLDT_SCENARIO_OBSERVATION_PAUSE,
    CLDT_SCENARIO_ENDPOINT_RESTART,
    CLDT_SCENARIO_TOPOLOGY_SHIFT
} cldt_scenario_kind_t;

/*
 * Labels remain strings until provisioning maps them to physical numeric node
 * IDs. Do not hash labels ad hoc: an implementation must reject an unknown
 * label or use one documented collision-checked mapping.
 */
typedef struct {
    char label[CLDT_MANIFEST_ID_BYTES];
    cldt_node_role_t role;
} cldt_manifest_node_t;

typedef struct {
    char id[CLDT_MANIFEST_ID_BYTES];
    char source_label[CLDT_MANIFEST_ID_BYTES];
    cldt_traffic_class_t traffic_class;
    uint32_t period_ms;
    uint16_t payload_bytes;
    uint32_t deadline_ms;
    uint16_t burst_packets;
} cldt_workload_config_t;

typedef struct {
    /*
     * run_id is assigned by the launcher only after parsing and cross-field
     * validation. It is never taken from a template. Before assignment, the
     * launcher reserves a nonzero cryptographically generated value in the
     * durable global run ledger and binds a non-secret command-key identity only
     * for an actuated run. The parser leaves it zero; the coordinator rejects zero.
     */
    char experiment_id[CLDT_MANIFEST_ID_BYTES];
    cldt_run_id_t run_id;
    /* Nonzero identity of the launcher process; assigned beside run_id. */
    cldt_boot_id_t command_authority_boot_id;
    uint32_t seed;

    cldt_manifest_node_t nodes[CLDT_MAX_MANIFEST_NODES];
    size_t node_count;
    uint8_t thread_channel;
    char placement[CLDT_MANIFEST_TEXT_BYTES];
    char firmware_reference[CLDT_MANIFEST_TEXT_BYTES];

    uint32_t warmup_s;
    uint32_t measurement_s;
    uint32_t cooldown_s;
    uint16_t repetitions;

    cldt_workload_config_t workloads[CLDT_MAX_WORKLOADS];
    size_t workload_count;

    cldt_scenario_kind_t scenario;
    uint32_t scenario_at_s;
    uint32_t scenario_duration_s;
    char scenario_target[CLDT_MANIFEST_TEXT_BYTES];

    cldt_treatment_mode_t treatment_mode;
    cldt_candidate_action_t candidate_action;
    /*
     * Identifier selected by treatment.control_profile. Parsing preserves this
     * bounded name only; the host registry must resolve it to a
     * cldt_control_profile_t, verify its digest, and record both identities in
     * the evidence bundle before coordinator initialization.
     */
    char control_profile[CLDT_MANIFEST_TEXT_BYTES];
    bool host_model_enabled;
    bool remote_actuation_enabled;

    bool counter_reconciliation_required;
    double minimum_critical_on_time_pdr;
    char negative_case[CLDT_MANIFEST_TEXT_BYTES];
    uint8_t canonical_digest[CLDT_CONFIG_DIGEST_BYTES];
} cldt_experiment_config_t;

/*
 * Parses exactly one UTF-8 ready manifest from caller-owned bytes.
 *
 * Implementation sequence:
 * - validate JSON syntax and schema first;
 * - reject state == "template" before allocating or opening I/O;
 * - copy bounded fields, preserving a precise error path;
 * - calculate canonical_digest after full validation and leave run_id plus
 *   command_authority_boot_id zero for the launcher's separate assignment step.
 *
 * The parser must reject unknown runtime fields, duplicate object keys, secrets,
 * and any null value that a ready manifest is required to replace.
 */
cldt_status_t cldt_experiment_config_parse(
    const char *json,
    size_t json_bytes,
    cldt_experiment_config_t *output);

/*
 * Performs deterministic cross-field validation after parsing.
 *
 * It verifies node/stream references, rate and deadline feasibility, scenario
 * timing, treatment permissions, and compiled safety ceilings. It must not make
 * network calls, create a directory, or mutate output state.
 */
cldt_status_t cldt_experiment_config_validate(
    const cldt_experiment_config_t *config);

#ifdef __cplusplus
}
#endif

#endif


### `host/experiment_config.c`

In [ ]:
%%writefile /content/cldt_scratch/experiment_config.c
#include "experiment_config.h"

cldt_status_t cldt_experiment_config_parse(
    const char *json,
    size_t json_bytes,
    cldt_experiment_config_t *output)
{
    (void)json;
    (void)json_bytes;
    (void)output;

    /*
     * IMPLEMENTATION TODO:
     * 1. Use a maintained JSON parser with a bounded input limit; parse exactly
     *    one UTF-8 document and reject duplicate keys rather than accepting a
     *    library-specific last-key-wins behavior.
     * 2. Validate against schemas/experiment.schema.json before conversion.
     *    This runtime parser accepts only state == "ready"; a template is a
     *    planning artifact and must never start a physical run.
     * 3. Copy strings into fixed, NUL-terminated fields only after checking the
     *    destination capacity. Require a non-empty control_profile for a ready
     *    run and preserve JSON Pointer-like error paths for the operator instead
     *    of returning a generic parse failure.
     * 4. Compute the canonical manifest digest from the original validated bytes
     *    using one documented canonicalization rule; do not include credentials.
     *    Leave output.run_id and output.command_authority_boot_id zero. The
     *    launcher assigns them only after a separate durable global-ledger
     *    reservation; parsing must not silently create nonce state or require a
     *    command key for a shadow-only run.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_experiment_config_validate(
    const cldt_experiment_config_t *config)
{
    (void)config;

    /*
     * IMPLEMENTATION TODO: validate cross-field relationships that schema shape
     * checks cannot prove: every stream source must name an endpoint, deadlines
     * must be compatible with their period, aggregate offered load must fit the
     * compiled safety ceiling, scenario time must lie inside measurement time,
     * and remote actuation must be disabled for non-control treatments. Version
     * one accepts only NONE or BULK_RATE_REDUCE and rejects the reserved phase
     * and power actions even though planning templates can name them. Reject a
     * configuration before any adapter or run directory is opened. Keep this
     * function deterministic so the same manifest has the same outcome on host
     * and in future gateway subset validation.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}


Pekerjaan load-step admission tetap mengikuti dua fungsi yang ada:

1. `cldt_experiment_config_parse()` hanya menerima strict JSON `state == "ready"`; JSONC dan template tidak pernah memulai run.
2. `cldt_experiment_config_validate()` menuntut `CLDT_SCENARIO_LOAD_STEP`, `CLDT_TREATMENT_PREDICTION`, `CLDT_ACTION_NONE`, `host_model_enabled == true`, dan `remote_actuation_enabled == false` untuk experiment ini.
3. `scenario_at_s` dan `scenario_duration_s` harus berada pada measured interval menurut satu reference rule yang terdokumentasi. Schema hanya mengecek range angka, bukan hubungan ini.
4. Stream source harus menunjuk endpoint yang ada; critical stream tetap frozen; aggregate offered rate dan deadline feasibility diperiksa sebelum recorder/broker.
5. Ready schema menerima beberapa string sampai 512 karakter, sedangkan runtime buffer hanya 161 bytes. Jalur paling kecil adalah memakai actual ready strings yang muat 160 bytes dan menolak input yang lebih panjang sebelum copy; schema-valid tidak berarti representable.
6. `calibration_id` berasal dari resolved profile. Current config tidak membawa baseline-evidence ID atau model freeze artifact; field lain tidak dioverload untuk tujuan itu.
7. Shadow run tetap memperoleh nonzero `run_id` dan coordinator boot identity dari Phase-3 launcher/ledger. Ia tidak memerlukan command key.

> **Admission stop:** bila before/during step, baseline binding, calibration split, atau horizon schedule masih hanya ada di catatan bebas, strict manifest belum cukup untuk menjalankan held-out comparison secara reproducible.

## Step 5: Five-State Kalman Numerical Core
### `host/kalman.h`

In [ ]:
%%writefile /content/cldt_scratch/kalman.h
#ifndef CLDT_HOST_KALMAN_H
#define CLDT_HOST_KALMAN_H

#include <stdbool.h>
#include <stdint.h>
#include "cldt/cldt_status.h"

#define CLDT_KALMAN_DIM 5

typedef struct {
    float x[CLDT_KALMAN_DIM];                          // state estimate
    float P[CLDT_KALMAN_DIM][CLDT_KALMAN_DIM];         // estimate covariance
    float F[CLDT_KALMAN_DIM][CLDT_KALMAN_DIM];         // state transition
    float H[CLDT_KALMAN_DIM][CLDT_KALMAN_DIM];         // observation model
    float Q[CLDT_KALMAN_DIM][CLDT_KALMAN_DIM];         // process noise
    float R[CLDT_KALMAN_DIM][CLDT_KALMAN_DIM];         // measurement noise
    uint64_t last_update_us;
    uint32_t update_count;
    bool initialized;
} cldt_kalman_t;

cldt_status_t cldt_kalman_init(cldt_kalman_t *kf);
cldt_status_t cldt_kalman_set_model(cldt_kalman_t *kf, const float F[5][5], const float H[5][5], const float Q[5][5], const float R[5][5]);
cldt_status_t cldt_kalman_predict(cldt_kalman_t *kf);
cldt_status_t cldt_kalman_update(cldt_kalman_t *kf, const float z[5]);
float cldt_kalman_state_uncertainty(const cldt_kalman_t *kf, int state_index);

#endif // CLDT_HOST_KALMAN_H


### Source TODO: `host/kalman.c`

In [ ]:
%%writefile /content/cldt_scratch/kalman.c
#include "kalman.h"
#include <string.h>

#define CLDT_KALMAN_INIT_VARIANCE 1000.0f

// TODO: Matrix helper signatures needed: mat5_multiply(out[5][5], a[5][5], b[5][5]), mat5_transpose, mat5_add, mat5_vec5_multiply, vec5_sub
// TODO: Gauss-Jordan 5x5 inverse: partial pivoting, scale pivot row, eliminate column, ~80 lines

cldt_status_t cldt_kalman_init(cldt_kalman_t *kf) {
    if (!kf) return CLDT_ERR_INVALID_ARGUMENT;

    memset(kf, 0, sizeof(cldt_kalman_t));
    for (int i = 0; i < 5; ++i) {
        kf->P[i][i] = CLDT_KALMAN_INIT_VARIANCE;
    }
    kf->initialized = false;
    return CLDT_OK;
}

cldt_status_t cldt_kalman_set_model(cldt_kalman_t *kf, const float F[5][5], const float H[5][5], const float Q[5][5], const float R[5][5]) {
    if (!kf || !F || !H || !Q || !R) return CLDT_ERR_INVALID_ARGUMENT;

    (void)kf;
    (void)F;
    (void)H;
    (void)Q;
    (void)R;

    // TODO: Q tuning: run pilot data with static network, compute empirical variance of prediction residuals
    // TODO: R tuning: collect readings while state is forced static, calculate empirical variance
    
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_kalman_predict(cldt_kalman_t *kf) {
    if (!kf || !kf->initialized) return CLDT_ERR_WRONG_STATE;

    (void)kf;

    // TODO: Exact Kalman equations: x_pred = F*x, P_pred = F*P*F^T + Q
    // TODO: Measure update cost on the actual host build; do not infer real-time behavior from an operation-count estimate

    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_kalman_update(cldt_kalman_t *kf, const float z[5]) {
    if (!kf || !kf->initialized || !z) return CLDT_ERR_WRONG_STATE;

    (void)kf;
    (void)z;

    // TODO: Exact Kalman equations: y = z - H*x, S = H*P*H^T + R, K = P*H^T*S^{-1}, x = x + K*y, P = (I-K*H)*P

    return CLDT_ERR_NOT_IMPLEMENTED;
}

float cldt_kalman_state_uncertainty(const cldt_kalman_t *kf, int state_index) {
    if (!kf || state_index < 0 || state_index >= 5) return -1.0f;
    
    (void)kf;
    (void)state_index;

    // TODO: P diagonal gives per-state uncertainty: P[2][2] = critical_pdr variance for fidelity gate

    return -1.0f;
}


Urutan numerical work:

1. State order dibekukan persis seperti `DESIGN.md`: endpoint A queue occupancy, endpoint B queue occupancy, critical PDR, parent link quality, dan MAC retry rate. Matrices yang memakai urutan lain tidak dicampur dengan revision ini.
2. File-local helper mengimplementasikan 5×5 multiplication, transpose, add, matrix-vector multiplication, vector subtraction, dan inverse dengan partial pivoting. Null, non-finite, near-singular pivot, dan failed inverse menghasilkan explicit failure; output state tidak dipublikasikan separuh.
3. `cldt_kalman_init()` hanya membuat deterministic empty state dan initial covariance. Initial state, `last_update_us`, serta kapan `initialized` menjadi true harus dibekukan pada contract; source sekarang belum menentukannya.
4. `cldt_kalman_set_model()` menyalin `F`, `H`, `Q`, dan `R` yang sudah dipilih dari calibration evidence. Pilot variance tidak dihitung tersembunyi di initializer.
5. `cldt_kalman_predict()` menjalankan (x^- = F x) dan (P^- = FPF^T + Q) pada local scratch, lalu commit hanya bila seluruh elemen finite.
6. `cldt_kalman_update()` menjalankan innovation, covariance, gain, state, dan covariance update sesuai komentar source. Input asynchronous/missing belum representable oleh signature `z[5]`; ownership itu ditutup sebelum fungsi dipakai pada physical stream.
7. `cldt_kalman_state_uncertainty()` mengembalikan diagonal `P[state_index][state_index]` hanya untuk initialized, finite, nonnegative covariance. `P[2][2]` dicatat sebagai uncertainty; Phase 4 belum menggunakannya untuk gate transition.
8. `F/H/Q/R`, initial covariance, calibration evidence identity, code revision, dan state order disimpan sebagai one frozen artifact. Refit menghasilkan artifact identity/digest baru; ini berbeda dari runtime `model_revision` yang source naikkan ketika accepted observation benar-benar mengubah state.

| Numerical verification | Contoh expected form |
|---|---|
| Null and uninitialized paths | e.g. exact non-success status |
| Identity `F` / `H`, zero-noise fixed case | e.g. exact state/covariance |
| Diagonal positive `Q` / `R` | e.g. finite expected vector |
| Singular innovation covariance | e.g. rejected; prior state unchanged |
| Non-finite matrix/input | e.g. rejected |
| Covariance symmetry tolerance | e.g. documented bound |
| `P[2][2]` | e.g. finite and nonnegative |
| Repeated deterministic input | e.g. byte/numeric result identical within frozen tolerance |

Repository belum mendaftarkan target test untuk `host/kalman.c`. Tanpa test owner yang nyata, table ini tetap verification plan dan bukan PASS claim.

## Step 6: Three Frozen Model Instances
### `host/twin_model.h`

In [ ]:
%%writefile /content/cldt_scratch/twin_model.h
#ifndef CLDT_HOST_TWIN_MODEL_H
#define CLDT_HOST_TWIN_MODEL_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "cldt/cldt_status.h"
#include "cldt/cldt_types.h"
#include "experiment_config.h"

#ifdef __cplusplus
extern "C" {
#endif

#define CLDT_MODEL_MAX_NODES 8U

typedef struct {
    cldt_model_variant_t variant;
    uint32_t queue_depth[CLDT_MODEL_MAX_NODES][CLDT_TRAFFIC_COUNT];
    uint32_t service_time_us[CLDT_MODEL_MAX_NODES][CLDT_TRAFFIC_COUNT];
    uint32_t link_rtt_us[CLDT_MODEL_MAX_NODES];
    double link_delivery_probability[CLDT_MODEL_MAX_NODES];
    uint64_t modeled_time_us;
    uint64_t model_revision;
    bool calibrated;
} cldt_twin_model_t;

typedef struct {
    cldt_model_variant_t model_variant;
    uint64_t model_revision;
    uint64_t issued_host_us;
    uint64_t horizon_start_us;
    uint64_t horizon_end_us;
    double predicted_pdr;
    uint32_t predicted_p50_rtt_us;
    uint32_t predicted_p95_rtt_us;
    double predicted_deadline_miss_ratio;
    double lower_interval;
    double upper_interval;
} cldt_prediction_t;

cldt_status_t cldt_twin_model_init(
    cldt_twin_model_t *model,
    const cldt_experiment_config_t *config,
    cldt_model_variant_t variant);

/* Updates state from one physical trace without performing policy selection. */
cldt_status_t cldt_twin_model_observe(
    cldt_twin_model_t *model,
    const cldt_trace_record_t *record);

/* Runs a side-effect-free what-if horizon under one candidate policy. */
cldt_status_t cldt_twin_model_predict(
    const cldt_twin_model_t *model,
    const cldt_policy_t *candidate,
    uint64_t issued_host_us,
    uint64_t horizon_start_host_us,
    uint64_t horizon_end_host_us,
    cldt_prediction_t *output);

#ifdef __cplusplus
}
#endif

#endif


### Source TODO: `host/twin_model.c`

In [ ]:
%%writefile /content/cldt_scratch/twin_model.c
#include "twin_model.h"

cldt_status_t cldt_twin_model_init(
    cldt_twin_model_t *model,
    const cldt_experiment_config_t *config,
    cldt_model_variant_t variant)
{
    (void)model;
    (void)config;
    (void)variant;

    /*
     * IMPLEMENTATION TODO: reject a null model, unvalidated configuration, or
     * invalid variant; reset every node/class state deterministically, record the
     * starting model revision, and leave calibrated false until the estimator
     * admits evidence. Initialize only state permitted by the variant's frozen
     * feature allowlist and needed to predict the declared primary metric. Do
     * not introduce a broad simulator, dashboard state, or policy state here.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_twin_model_observe(
    cldt_twin_model_t *model,
    const cldt_trace_record_t *record)
{
    (void)model;
    (void)record;

    /*
     * IMPLEMENTATION TODO: validate the physical record, use node ID plus boot
     * ID and sequence to reject duplicates or old observations, and update only
     * the state fields justified by that event kind. Advance modeled_time_us
     * monotonically; an out-of-order record may be retained by the recorder but
     * must not roll model state backward. Increment model_revision only when a
     * logical state change is accepted and record enough context to audit it.
     * The naive, network-only, and cross-layer instances receive the same
     * eligible horizon boundaries but update only from their declared features.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_twin_model_predict(
    const cldt_twin_model_t *model,
    const cldt_policy_t *candidate,
    uint64_t issued_host_us,
    uint64_t horizon_start_host_us,
    uint64_t horizon_end_host_us,
    cldt_prediction_t *output)
{
    (void)model;
    (void)candidate;
    (void)issued_host_us;
    (void)horizon_start_host_us;
    (void)horizon_end_host_us;
    (void)output;

    /*
     * IMPLEMENTATION TODO: require a calibrated model, a validated candidate,
     * and issued <= horizon_start < horizon_end with overflow-safe duration
     * bounds. Copy the live model to local scratch state,
     * evolve only that copy, and write a prediction containing horizon end,
     * service outcome, and uncertainty interval. Populate variant, model
     * revision, issuance time, and exact start/end boundaries so the estimator
     * can score all variants against the same future window without leakage.
     * Never select or transmit policy from this function.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}


Pekerjaan model dibagi sesuai tiga fungsi:

1. `cldt_twin_model_init()` menolak invalid config/variant, membersihkan state secara deterministik, memilih starting revision, dan mempertahankan `calibrated == false`. Ia tidak memasukkan dashboard, broker, gate, atau policy state.
2. Model naive memakai historical moving average yang frozen. Network-only dan cross-layer memakai model family, loss, calibration block, and horizon schedule yang sama; naive adalah benchmark terpisah, bukan family yang dipaksa sama.
3. Network-only memakai pre-issuance delivery outcome, `link_rssi_dbm`, dan offered traffic load. Cross-layer memakai set itu plus exact MAC counters, queue/deadline state, parent link quality, dan FreeRTOS trace yang mapping-nya sudah ditutup Phase 3.
4. `cldt_twin_model_observe()` harus menolak duplicate/old identity, menghindari rollback time, dan mengubah hanya state yang diizinkan variant. Current struct belum mempunyai boot/sequence mapping; header diselesaikan sebelum body.
5. Calibration observation boleh memperbarui fitted state. Held-out observation tidak boleh memanggil learning update; ia hanya membentuk observed outcome untuk score.
6. `cldt_twin_model_predict()` menyalin model ke scratch, memakai only data available by `issued_host_us`, dan mengisi variant, runtime state revision, issue time, exact horizon, point estimates, serta interval. Frozen artifact identity tetap disimpan terpisah pada evidence bundle.
7. Prediction treatment memakai no candidate action. Signature masih meminta `candidate`; semantics baseline policy snapshot versus nullable candidate harus ditutup satu kali—bukan dipilih berbeda per variant.
8. Prediction dari ketiga variant disimpan sebelum outcome horizon tersedia. Setelah outcome terlihat, captured runtime `model_revision` dan bytes prediction itu tidak ditulis ulang; prediction berikutnya boleh membawa revision lebih tinggi hanya bila accepted pre-issuance observation mengubah state sesuai source contract.

| Parity check | Contoh hasil |
|---|---|
| Same calibration run identities | e.g. PASS |
| Same loss for network/cross | e.g. PASS |
| Same issue/start/end horizons | e.g. PASS untuk setiap paired row |
| Network model sees cross-layer field | e.g. 0 occurrences / rejected test |
| Held-out update count | e.g. 0 |
| Frozen artifact digest changes after freeze | e.g. 0 |
| Runtime `model_revision` transition | e.g. naik hanya pada accepted logical state change; exact value captured per prediction |
| Prediction interval target and units | e.g. critical deadline-delivery ratio, proportion `[0,1]`, machine-bound in the header contract |
| Prediction interval ordered | e.g. `lower_interval <= bound predicted target <= upper_interval` |
| Side-effect-free prediction | e.g. input model bytes unchanged |

## Step 7: Calibration Update dan Horizon Scoring
### `host/estimator.h`

In [ ]:
%%writefile /content/cldt_scratch/estimator.h
#ifndef CLDT_HOST_ESTIMATOR_H
#define CLDT_HOST_ESTIMATOR_H

#include <stdint.h>

#include "cldt/cldt_status.h"
#include "cldt/cldt_metrics.h"
#include "cldt/cldt_types.h"
#include "twin_model.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef struct {
    uint64_t accepted_observations;
    uint64_t rejected_observations;
    uint64_t last_observation_host_us;
    double residual_mean;
    double residual_m2;
} cldt_estimator_t;

cldt_status_t cldt_estimator_init(cldt_estimator_t *estimator);

/* Updates model parameters, never policy or gate state. */
cldt_status_t cldt_estimator_observe(
    cldt_estimator_t *estimator,
    cldt_twin_model_t *model,
    const cldt_trace_record_t *record,
    uint64_t received_host_us);

/* Compares a prior prediction with observations from the same horizon. */
cldt_status_t cldt_estimator_score_prediction(
    cldt_estimator_t *estimator,
    const cldt_prediction_t *prediction,
    const cldt_metrics_t *observed,
    cldt_fidelity_sample_t *output);

#ifdef __cplusplus
}
#endif

#endif


### Source TODO: `host/estimator.c`

In [ ]:
%%writefile /content/cldt_scratch/estimator.c
#include "estimator.h"

cldt_status_t cldt_estimator_init(cldt_estimator_t *estimator)
{
    (void)estimator;

    /*
     * IMPLEMENTATION TODO: reject a null estimator, clear accepted/rejected
     * observation counts, last-observation time, residual mean, and residual M2
     * using a numerically stable initial state. Estimator selection and tuning
     * belong in a documented calibration block, not hidden in init.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_estimator_observe(
    cldt_estimator_t *estimator,
    cldt_twin_model_t *model,
    const cldt_trace_record_t *record,
    uint64_t received_host_us)
{
    (void)estimator;
    (void)model;
    (void)record;
    (void)received_host_us;

    /*
     * IMPLEMENTATION TODO: verify every pointer and host receive time, reject
     * records that predate the accepted sequence/boot state or violate the
     * configured observation-age limit, and increment rejected_observations
     * without modifying the model. For accepted records, update only fitted
     * parameters permitted by the calibration plan and record the receive time.
     * Do not learn from held-out runs when evaluating the primary comparison.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_estimator_score_prediction(
    cldt_estimator_t *estimator,
    const cldt_prediction_t *prediction,
    const cldt_metrics_t *observed,
    cldt_fidelity_sample_t *output)
{
    (void)estimator;
    (void)prediction;
    (void)observed;
    (void)output;

    /*
     * IMPLEMENTATION TODO: require that observed metrics are complete and refer
     * to the same run, policy epoch, and horizon as prediction. Compute residual
     * and interval coverage with a documented formula, update online statistics
     * only once per scored horizon, and populate every fidelity-sample field,
     * including model variant/revision and exact horizon identity.
     * A missing, stale, or unreconciled observation must produce a gate-relevant
     * failure signal rather than a favorable residual.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}


Urutan estimator work:

1. `cldt_estimator_init()` membuat zero-count, zero-time, dan stable residual statistics tanpa memilih tuning.
2. `cldt_estimator_observe()` membedakan accepted/rejected observation tanpa memodifikasi model pada rejection. Current struct/signature tidak memiliki configured age limit atau calibration/frozen mode; contract tersebut ditambahkan pada owner sebelum call path.
3. Online residual statistics memakai numerically stable update. Untuk residual (e_n), mean dan `residual_m2` diperbarui dengan two-delta/Welford form, bukan sum-of-squares yang mudah kehilangan presisi.
4. `cldt_estimator_score_prediction()` baru berjalan setelah exact horizon selesai, source complete, raw lifecycle audit lulus, aggregate reconcile, dan run/epoch/horizon identity cocok.
5. Primary target adalah critical deadline-delivery outcome pada horizon yang sama. Relative denominator rule untuk observed value dekat nol dibekukan sebelum held-out bytes dibuka. Karena current prediction struct mempunyai beberapa point metric tetapi hanya satu interval pair, header contract harus mengikat interval tersebut ke exact target dan units; scorer belum sah mengasumsikannya dari urutan field.
6. Interval coverage adalah satu bila observed value untuk metric yang terikat tadi berada di inclusive frozen interval dan nol bila di luar; aggregate coverage dilaporkan lintas eligible horizons. Prediction tanpa target/units binding dinyatakan malformed, bukan ditebak.
7. `cldt_fidelity_sample_t` diisi lengkap, termasuk integrity/support status. Phase 4 menyimpan record tersebut tetapi tidak memberikannya ke `cldt_fidelity_gate_evaluate()`.
8. Satu prediction/horizon/variant hanya dinilai sekali. Duplicate evaluation tidak menaikkan `sample_count` atau mengubah residual dua kali.

Critical mismatch saat ini: `cldt_estimator_score_prediction()` meminta same run, policy epoch, and horizon, tetapi input `cldt_prediction_t` dan `cldt_metrics_t` belum memuat seluruh identity tersebut. Body scorer belum dapat diimplementasikan jujur sampai observed-horizon contract diperbaiki.

| Scoring check | Contoh result |
|---|---|
| Prediction before horizon | e.g. accepted for pending storage |
| Outcome before horizon complete | e.g. not scored |
| Wrong run/epoch/horizon | e.g. rejected |
| Unreconciled outcome | e.g. integrity false; no favorable residual |
| Duplicate evaluation | e.g. rejected |
| Near-zero denominator | e.g. handled by frozen rule |
| Interval lower/upper violation | e.g. rejected as malformed prediction |
| Held-out fitted-parameter mutation | e.g. 0; runtime state/revision tetap boleh maju dari accepted pre-issuance observation |

## Step 8: Shadow-Only Coordinator
### `host/coordinator.h`

In [ ]:
%%writefile /content/cldt_scratch/coordinator.h
#ifndef CLDT_HOST_COORDINATOR_H
#define CLDT_HOST_COORDINATOR_H

#include <stdbool.h>
#include <stdint.h>

#include "broker_io.h"
#include "cldt/cldt_control_profile.h"
#include "estimator.h"
#include "experiment_config.h"
#include "fidelity_gate.h"
#include "policy.h"
#include "run_recorder.h"
#include "twin_model.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef struct {
    cldt_experiment_config_t config;
    /*
     * Resolved profile copied only after its ID matches config.control_profile
     * and its immutable digest has been recorded by the run recorder.
     */
    cldt_control_profile_t control_profile;
    cldt_broker_io_t broker;
    cldt_run_recorder_t recorder;
    /* One frozen instance and estimator per declared comparison variant. */
    cldt_twin_model_t models[CLDT_MODEL_VARIANT_COUNT];
    cldt_estimator_t estimators[CLDT_MODEL_VARIANT_COUNT];
    cldt_fidelity_gate_t gate;
    cldt_policy_t active_policy;
    uint64_t started_host_us;
    bool stop_requested;
} cldt_coordinator_t;

cldt_status_t cldt_coordinator_init(
    cldt_coordinator_t *coordinator,
    const cldt_experiment_config_t *config,
    const cldt_control_profile_t *control_profile);

cldt_status_t cldt_coordinator_run(cldt_coordinator_t *coordinator);

void cldt_coordinator_request_stop(cldt_coordinator_t *coordinator);

#ifdef __cplusplus
}
#endif

#endif


### Source TODO: `host/coordinator.c`

In [ ]:
%%writefile /content/cldt_scratch/coordinator.c
#include "coordinator.h"

cldt_status_t cldt_coordinator_init(
    cldt_coordinator_t *coordinator,
    const cldt_experiment_config_t *config,
    const cldt_control_profile_t *control_profile)
{
    (void)coordinator;
    (void)config;
    (void)control_profile;

    /*
     * IMPLEMENTATION TODO:
     * 1. Require a ready, cross-field-validated config with a nonzero run ID
     *    already reserved in the durable global run ledger and a nonzero command
     *    authority boot ID, plus a valid resolved control profile. Compare the
     *    profile ID with config.control_profile
     *    using bounded strings; reject mismatch before opening a recorder or
     *    broker. The caller is responsible for checking the profile digest
     *    against the canonical registry document before this function is called.
     * 2. Copy config and profile only after validation. Record the profile ID,
     *    calibration ID, actuation-model variant, and digest beside the frozen
     *    manifest. Version one may name only the cross-layer variant for
     *    actuation; failed shadow acceptance means no actuation, not model swap.
     *    Initialize the fidelity gate and edge proposal limits from that
     *    immutable selection.
     * 3. Initialize recorder, all three model/estimator pairs, fidelity gate,
     *    policy baseline, and broker in that order. Each successful step needs a
     *    paired rollback action so a later failure leaves no partial run marked
     *    valid.
     * 4. Do not open a network adapter before the immutable run directory,
     *    manifest digest, and control-profile identity exist.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_coordinator_run(cldt_coordinator_t *coordinator)
{
    (void)coordinator;

    /*
     * IMPLEMENTATION TODO: implement one bounded event loop with this strict
     * sequence for each accepted observation: record raw bytes first; validate
     * run/digest identity; update each model only from its allowed features;
     * score all variants on identical completed prior horizons; evaluate the
     * fidelity gate only for the frozen actuation variant; and only then consider
     * a new bounded proposal. Sleep or poll with a deadline so policy expiry,
     * phase transitions, and stop requests are never starved by broker traffic.
     * Finalize through the recorder with complete, invalid, or interrupted status.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

void cldt_coordinator_request_stop(cldt_coordinator_t *coordinator)
{
    (void)coordinator;

    /*
     * IMPLEMENTATION TODO: make this function only set an atomic or signal-safe
     * stop flag. The main loop owns broker close, final counter requests, metric
     * reconciliation, and recorder finalization because those operations may
     * allocate, block, or fail. A signal handler must never write evidence or
     * publish a fallback command directly.
     */
}


### Current host build boundary: `host/CMakeLists.txt`

In [ ]:
%%writefile /content/cldt_scratch/host_CMakeLists.txt
add_executable(cldt_host
    main.c
    coordinator.c
    experiment_config.c
    twin_model.c
    estimator.c
    fidelity_gate.c
    policy.c
    broker_io.c
    run_recorder.c
    kalman.c
)

target_include_directories(cldt_host PRIVATE ${CMAKE_CURRENT_SOURCE_DIR})
target_link_libraries(cldt_host PRIVATE cldt_common)

if(MSVC)
    target_compile_options(cldt_host PRIVATE /W4)
else()
    target_compile_options(cldt_host PRIVATE -Wall -Wextra -Wpedantic -Wconversion)
endif()


Shadow coordinator sequence:

1. `cldt_coordinator_init()` memakai ready validated config, nonzero reserved run identity, resolved profile/calibration identity, recorder, broker, and three model/estimator pairs. Phase 4 branch rejects `remote_actuation_enabled == true`.
2. Model matrices, feature contracts, calibration/frozen-artifact digest, horizon schedule, dan metric-bound interval rule sudah frozen sebelum broker dibuka untuk held-out run. Runtime `model_revision` bukan artifact ID; nilainya ikut dicatat pada setiap prediction snapshot.
3. Broker callback kembali cepat. Raw inbound bytes masuk recorder lebih dahulu; validation/decode/model work terjadi pada bounded event-loop owner.
4. Accepted observation dikirim ke tiga variant dengan allowlist berbeda. A model may reject a record without making the raw line disappear.
5. Prediction schedule membuat three predictions untuk exact same future horizon and stores them in bounded pending ownership.
6. Completed prior horizon is aggregated/reconciled once, then all three predictions are scored. Held-out observations never call calibration update.
7. `fidelity_gate` and `policy` members may exist in struct/build but Phase 4 branch does not evaluate gate or generate/publish proposal. Link inclusion is not runtime authorization.
8. Poll/deadline prevents pending horizon, phase transition, stop, cooldown, and finalization starvation.
9. Stop request only changes stop intent. Main loop closes measurement, captures final counters, resolves or marks pending horizons, reconciles, and writes one terminal status.
10. Any buffer overflow, identity mismatch, missing source, or unresolved horizon stays visible and classifies the run under the frozen rule.

Current `cldt_coordinator_t` has no bounded inbound queue or pending prediction store. `host/CMakeLists.txt` also declares no JSON/MQTT/numerical dependency and `tests/CMakeLists.txt` has no host model test. Those are explicit implementation blockers, not reasons to call the loop complete.

### Current registered test surface: `tests/CMakeLists.txt`

In [ ]:
%%writefile /content/cldt_scratch/tests_CMakeLists.txt
function(cldt_add_skeletal_test name source)
    add_executable(${name} ${source})
    target_link_libraries(${name} PRIVATE cldt_common)
    add_test(NAME ${name} COMMAND ${name})
    set_tests_properties(${name} PROPERTIES SKIP_RETURN_CODE 77)
endfunction()

cldt_add_skeletal_test(test_protocol test_protocol.c)
cldt_add_skeletal_test(test_crc32c test_crc32c.c)
cldt_add_skeletal_test(test_auth test_auth.c)
cldt_add_skeletal_test(test_clock_sync test_clock_sync.c)
cldt_add_skeletal_test(test_metrics test_metrics.c)
cldt_add_skeletal_test(test_event_trace test_event_trace.c)
cldt_add_skeletal_test(test_control_profile test_control_profile.c)


The seven registered executables cover common-library contracts only. A Week-4 freeze needs deterministic checks for Kalman matrices, model ordering, feature isolation, calibration freeze, horizon matching, and shadow coordinator behavior. Karena file/target tersebut belum ada, notebook tidak memberi nama test fiktif. Penambahan test owner adalah repo change yang harus disetujui dan tercatat sebelum closure, bukan PASS checkbox yang diisi dari notebook.

## Step 9: Physical Load-Step Ownership
### Existing Endpoint Workload Contract

Prediction run tidak boleh menyelundupkan command path. Existing workload API diperiksa karena scenario harus benar-benar mengubah offered bulk load pada waktu yang predeclared.

In [ ]:
%%writefile /content/cldt_scratch/workload.h
#ifndef CLDT_ENDPOINT_WORKLOAD_H
#define CLDT_ENDPOINT_WORKLOAD_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "esp_err.h"
#include "freertos/FreeRTOS.h"
#include "freertos/task.h"
#include "freertos/timers.h"

#include "cldt/cldt_types.h"
#include "deadline_queue.h"

#ifdef __cplusplus
extern "C" {
#endif

#define CLDT_ENDPOINT_MAX_STREAMS 8U

typedef struct {
    uint32_t stream_id;
    cldt_traffic_class_t traffic_class;
    uint32_t period_ms;
    uint32_t phase_ms;
    uint32_t jitter_ms;
    uint32_t deadline_ms;
    uint16_t payload_bytes;
    uint16_t burst_packets;
    uint16_t maximum_rate_pps;
} cldt_stream_config_t;

typedef struct {
    cldt_deadline_queue_t *queue;
    cldt_stream_config_t streams[CLDT_ENDPOINT_MAX_STREAMS];
    size_t stream_count;
    cldt_policy_t active_policy;
    TaskHandle_t producer_task;
    TimerHandle_t release_timer;
    uint32_t random_state;
    bool running;
} cldt_workload_t;

esp_err_t cldt_workload_init(
    cldt_workload_t *workload,
    cldt_deadline_queue_t *queue,
    const cldt_stream_config_t *streams,
    size_t stream_count,
    uint32_t seed);

/* Starts release timers only after the run digest is accepted. */
esp_err_t cldt_workload_start(cldt_workload_t *workload, uint64_t run_start_local_us);

/* Applies a prevalidated immutable policy snapshot at a release boundary. */
esp_err_t cldt_workload_apply_policy(
    cldt_workload_t *workload,
    const cldt_policy_t *policy);

/* ISR entry: capture no payload and wake only the producer task. */
void cldt_workload_event_isr(void *context);

esp_err_t cldt_workload_stop(cldt_workload_t *workload);

#ifdef __cplusplus
}
#endif

#endif


### Existing source TODO: `firmware/endpoint/main/workload.c`

In [ ]:
%%writefile /content/cldt_scratch/workload.c
#include "workload.h"

esp_err_t cldt_workload_init(
    cldt_workload_t *workload,
    cldt_deadline_queue_t *queue,
    const cldt_stream_config_t *streams,
    size_t stream_count,
    uint32_t seed)
{
    (void)workload;
    (void)queue;
    (void)streams;
    (void)stream_count;
    (void)seed;

    /*
     * IMPLEMENTATION TODO: validate non-null arguments, stream count, unique
     * stream IDs, payload/deadline/rate bounds, and aggregate offered rate against
     * the endpoint safety limit. Copy the approved stream list into workload-owned
     * storage, seed a documented deterministic jitter generator, and create the
     * producer task and timer with static allocation. A failed init must leave
     * running false and must not alter the deadline queue.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_workload_start(cldt_workload_t *workload, uint64_t run_start_local_us)
{
    (void)workload;
    (void)run_start_local_us;

    /*
     * IMPLEMENTATION TODO: require an accepted run start time and inactive
     * workload, calculate each first release from the same local monotonic epoch
     * plus its phase, and schedule absolute release intent rather than chaining
     * relative delays that accumulate jitter. Timer callbacks only notify the
     * producer task; payload creation, queue admission, tracing, and networking
     * happen in task context. Record release jitter against the intended time.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_workload_apply_policy(
    cldt_workload_t *workload,
    const cldt_policy_t *policy)
{
    (void)workload;
    (void)policy;

    /*
     * IMPLEMENTATION TODO: accept only a policy already authenticated and checked
     * by endpoint runtime, copy it into a staging snapshot, and swap it at one
     * documented release boundary so no stream sees half old/half new fields.
     * Revalidate that critical periods and reserved queue capacity remain inside
     * compiled limits. Trace old epoch, new epoch, and effective local time; do
     * not dynamically allocate or edit the manifest at runtime.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

void cldt_workload_event_isr(void *context)
{
    (void)context;

    /*
     * IMPLEMENTATION TODO: keep this ISR to the minimum allowed by FreeRTOS:
     * validate the stored context if practical, call the appropriate FromISR task
     * notification primitive, capture whether a higher-priority task woke, and
     * request a yield through the documented port macro. Do not allocate, log,
     * acquire a mutex, encode a frame, or call OpenThread from this ISR.
     */
}

esp_err_t cldt_workload_stop(cldt_workload_t *workload)
{
    (void)workload;

    /*
     * IMPLEMENTATION TODO: stop or disarm release timers, signal producer task
     * to stop creating new work, wait a bounded time for transport-owned slots,
     * explicitly expire remaining queued work if the deadline passes, and report
     * a final accounting snapshot. Only then set running false. Preserve the
     * reason for forced expiry so a fast shutdown never becomes invisible loss.
     */
    return ESP_ERR_NOT_SUPPORTED;
}


Current contract hanya mempunyai one static stream list dan `cldt_workload_apply_policy()`. Fungsi apply menerima policy yang sudah authenticated dan prevalidated; memakainya untuk hidden Week-4 remote command akan melanggar `remote_actuation == false`.

Sebelum physical load-step run:

1. Before-step stream shape dan during-step stream shape mempunyai one machine-readable owner.
2. `scenario.at_s` mempunyai exact reference—measurement start atau run start—dan mapping ke endpoint monotonic time.
3. Hanya bulk period, payload, atau burst yang berubah. Critical stream, channel, placement, firmware, and topology tetap.
4. Step berlaku pada one release boundary, memiliki recorded effective time, target stream identity, and acknowledgement.
5. `scenario.duration_s` menentukan restoration boundary; recovery tetap terlihat sebelum cooldown.
6. Failed scheduling, missed boundary, wrong target, or partial update membuat run invalid.
7. Local scenario path tidak menerima broker command and does not increment policy epoch as if a controller acted.
8. Pilots yang dipakai memilih recoverable disturbance tidak masuk final held-out tuning/evaluation set.

Current `workload.h`, `experiment_config.h`, dan schema belum menghubungkan these semantics end to end. Bila contract itu belum ditutup, operator tidak mengganti traffic manual di tengah run lalu menamainya reproducible load step. Manifest tetap template.

## Step 10: Reproduction Pipeline
### Exact Scaffold: `host/analysis/reproduce.py`

In [ ]:
%%writefile /content/cldt_scratch/reproduce.py
import sys
import json
from pathlib import Path

def main():
    if len(sys.argv) != 2:
        print("Usage: python reproduce.py <results_dir>")
        sys.exit(1)
        
    # TODO: load manifest JSON from results_dir / "manifest.json"
    # TODO: verify manifest has state="ready" and all _todo items resolved
    # TODO: compute SHA-256 digest of manifest and compare against results_dir / "manifest.sha256"
    # TODO: load events.ndjson one JSON object per line; reject malformed, blank,
    # duplicate, or trailing non-JSON records
    # TODO: group events by (run_id, node_id, boot_id, sequence) for per-item lifecycle audit
    # TODO: for each lifecycle group, verify exactly one release event and one terminal event (ack/expire/drop)
    # TODO: count duplicate_releases, duplicate_terminals, terminal_without_release, unresolved_items
    # TODO: fit naive moving-average baseline on calibration data only
    # TODO: fit network-only model: features = [delivery_outcome, link_rssi, traffic_load]
    # TODO: fit cross-layer model from the frozen network, MAC, queue, and RTOS
    # feature allowlist
    # TODO: read manifest-defined calibration and held-out blocks; never invent a percentage split after seeing results
    # TODO: score all three models on identical held-out horizons: relative P95 error on deadline delivery ratio
    # TODO: compute primary uncertainty from run-level summaries or a whole-run cluster bootstrap
    # TODO: use within-run block bootstrap only for paired time-series uncertainty,
    # never as independent physical replication
    # TODO: compute prediction interval coverage: fraction of observations within predicted +/- 2 sigma
    # TODO: build a calibration-only support envelope and retain inside/outside status for every scored horizon
    # TODO: retain observation-integrity status; missing/stale/unreconciled horizons must not disappear silently
    # TODO: perform feature-group ablation only after the primary three-model comparison is frozen
    # TODO: generate gate characterization: state/reason vs time, trust fraction,
    # false trust, abstention/requalification latency, and P[2][2]
    # TODO: output the frozen primary metric table as CSV
    # TODO: exit nonzero if reconciliation fails (any lifecycle inconsistency)
    # TODO: use numpy for statistics, matplotlib for plots, scipy.stats for bootstrap

    print("ERROR: reproduction pipeline is a scaffold and produced no result.", file=sys.stderr)
    raise SystemExit(2)

if __name__ == "__main__":
    main()


Pembagian TODO pada file tersebut:

| Kelompok | TODO |
|---|---|
| Phase-3 preflight yang tetap wajib | manifest/digest, strict ready state, NDJSON parsing, logical lifecycle grouping, duplicate/unresolved accounting, nonzero exit on reconciliation failure |
| Phase-4 primary | naive fit, matched network/cross fit, calibration versus held-out binding, identical horizon score, run-level/whole-run uncertainty, paired within-run block uncertainty, interval coverage, support/integrity status, frozen primary CSV |
| Ditunda | feature-group ablation sampai primary comparison frozen; gate characterization sampai fidelity gate Week 5 |

Urutan analysis:

1. Input contract harus dapat menunjuk seluruh calibration runs dan seluruh held-out runs. Current CLI menerima satu `results_dir`, sedangkan comments meminta blocks; directory layout/binding belum didefinisikan dan tidak boleh ditebak dari nama folder.
2. Manifest bytes and digest diverifikasi before loading derived table.
3. Every raw line is parsed once; malformed/blank/trailing records make the run invalid according to frozen rule.
4. Item audit and aggregate reconciliation run before model fit/score.
5. Naive fit uses calibration only. Network and cross use same model family, loss, calibration runs, horizons, fit procedure, and random seed plan; only feature allowlist differs.
6. Feature normalization and interval construction use calibration-only statistics.
7. Prediction table is generated before joining held-out outcome. Join uses exact run/horizon identity, not row order or nearest timestamp.
8. Primary summary treats independent physical runs as experimental units. Whole-run cluster bootstrap is primary; 1,000 within-run block resamples are secondary paired time-series uncertainty and never inflate replication.
9. Output records all valid, invalid, interrupted, missing, and integrity-failed horizons. No “dropna then score” shortcut.
10. CSV and figures are derived artifacts. Raw evidence/manifests remain immutable.

The file comments name `numpy`, `matplotlib`, and `scipy.stats`, but repository has no dependency or lock artifact. Reproducibility is not claimed until exact versions/environment are recorded by an owner selected in the repo.

## Step 11: Load-Step Manifest Authoring
### `experiments/authoring/load-step.jsonc`

JSONC berikut tetap planning copy. `null` dibiarkan sampai pilot/contract menghasilkan value. Comment dan `_todo` tidak dikirim ke runtime.

In [ ]:
%%writefile /content/cldt_scratch/load-step.jsonc
{
  // This authoring copy plans a held-out prediction experiment, never a controller test.
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "load-step-prediction",
  "title": "Held-Out Load-Step Prediction",
  "purpose": {
    "question": "Can a cross-layer model predict critical deadline degradation when a known bulk load step is introduced?",
    "comparison": "Naive, network-only, and cross-layer predictions on identical horizons from a load pattern not used to tune the models or gate.",
    "primary_metric": "Held-out P95 prediction error with run-aware uncertainty; gate trust metrics remain secondary."
  },
  "setup": {
    // Copy the completed stable baseline topology exactly and cite its evidence-bundle identifier.
    "nodes": null,
    "thread_channel": null,
    "placement": null,
    "firmware_reference": null
  },
  "execution": {
    // Keep the same phase structure as the baseline unless a predeclared reason requires change.
    "warmup_s": null,
    "measurement_s": null,
    "cooldown_s": null,
    "repetitions": null,
    "seed": null
  },
  "traffic": {
    // Freeze the critical stream; alter only predeclared bulk period, payload, or burst to form the load step.
    "streams": null
  },
  "scenario": {
    // Use load_step after warm-up and inside the measured window; identify the affected bulk stream.
    "event": null,
    "at_s": null,
    "duration_s": null,
    "target": null
  },
  "treatment": {
    // Required: prediction mode, host_model true, remote_actuation false, candidate_action none.
    "mode": null,
    "candidate_action": null,
    // Name the calibration/profile identity frozen before held-out scoring.
    "control_profile": null,
    "host_model": null,
    "remote_actuation": null
  },
  "acceptance": {
    "counter_reconciliation": null,
    // Carry forward the baseline critical-service floor rather than loosening it after the step.
    "minimum_critical_on_time_pdr": null,
    // Define an invalid case such as model code or physical topology changing during the held-out block.
    "negative_case": null
  },
  "evidence": {
    // Include calibration/held-out split, all three predictions, calibrated-region and observation-integrity status, raw traces, and model revision.
    "required_artifacts": null,
    "operator_notes_required": null,
    "topology_photo_required": null
  },
  "_todo": [
    {
      "path": "/setup",
      "action": "Bind this run to one completed baseline block.",
      "method": "Reuse board identities, channel, placement, and firmware; declare a new calibration block if any one changes.",
      "done_when": "No network or firmware change is hidden inside the prediction condition."
    },
    {
      "path": "/traffic",
      "action": "Create one recoverable bulk load step.",
      "method": "Use pilots outside the final set until critical behavior changes measurably but queues and radios recover.",
      "done_when": "Step time, duration, stream values, seed, and repetitions are frozen before held-out analysis."
    },
    {
      "path": "/treatment",
      "action": "Keep the run strictly shadow-only.",
      "method": "Score all three models on identical future horizons and prohibit model or gate tuning from held-out observations.",
      "done_when": "No remote policy can be issued in this condition."
    }
  ]
}


Field work memakai only keys yang memang ada:

| Field repo | Bentuk sumber nilai — bukan prefilled result |
|---|---|
| `setup.nodes` | exact four board labels/roles dari archived stable baseline |
| `setup.thread_channel` | channel yang sama dengan calibration block |
| `setup.placement` | copied physical placement/orientation/power path, actual string ≤160 bytes untuk runtime buffer |
| `setup.firmware_reference` | reference/digest pendek ke `versions.json` yang muat ≤160 bytes; full source/ESP-IDF/upstream identities dan four binary hashes berada pada artifact itu |
| `execution.warmup_s` | chosen from non-reportable pilots, schema 5–300 |
| `execution.measurement_s` | long enough to include pre-step, step, and recovery, schema 30–3600 |
| `execution.cooldown_s` | bounded drain/final counters, schema 5–300 |
| `execution.repetitions` | frozen before final outcomes; schema 1–30 |
| `execution.seed` | frozen uint32 plan |
| `traffic.streams` | concrete critical + bulk stream values within period/payload/deadline/burst bounds |
| `scenario.event` | exactly `load_step` |
| `scenario.at_s` | after warm-up and inside measured window under frozen time reference |
| `scenario.duration_s` | recoverable disturbance with recovery visible |
| `scenario.target` | existing bulk stream identity; actual string ≤160 bytes |
| `treatment.mode` | exactly `prediction` |
| `treatment.candidate_action` | exactly `none` |
| `treatment.control_profile` | exact registry profile that resolves to frozen `calibration_id` |
| `treatment.host_model` | exactly `true` |
| `treatment.remote_actuation` | exactly `false` |
| `acceptance.counter_reconciliation` | exactly `true` |
| `acceptance.minimum_critical_on_time_pdr` | unchanged baseline service floor |
| `acceptance.negative_case` | predeclared invalidation such as model code or topology change |
| `evidence.required_artifacts` | 4–8 unique artifact labels including raw traces, model/horizon records, reconciliation, and terminal status |
| `evidence.operator_notes_required` | chosen before run; true is the defensible physical protocol |
| `evidence.topology_photo_required` | copied/frozen rule for the physical block |

The template prose asks for a specific baseline evidence bundle and a calibration/held-out split, but neither has a dedicated schema key. `traffic.streams` also cannot unambiguously encode both before and during values. Its phrase `model revision` cannot honestly stand for both the immutable calibration/model artifact and the runtime state generation captured by each prediction. Existing unrelated fields are not overloaded to hide these gaps. The machine contract must be repaired before promotion.

### Strict runtime template: `experiments/load-step.json`

In [ ]:
%%writefile /content/cldt_scratch/load-step.json
{
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "load-step-prediction",
  "title": "Held-Out Load-Step Prediction",
  "purpose": {
    "question": "Can a cross-layer model predict critical deadline degradation when a known bulk load step is introduced?",
    "comparison": "Naive, network-only, and cross-layer predictions on identical horizons from a load pattern not used to tune the models or gate.",
    "primary_metric": "Held-out P95 prediction error with run-aware uncertainty; gate trust metrics remain secondary."
  },
  "setup": {
    "nodes": null,
    "thread_channel": null,
    "placement": null,
    "firmware_reference": null
  },
  "execution": {
    "warmup_s": null,
    "measurement_s": null,
    "cooldown_s": null,
    "repetitions": null,
    "seed": null
  },
  "traffic": { "streams": null },
  "scenario": { "event": null, "at_s": null, "duration_s": null, "target": null },
  "treatment": { "mode": null, "candidate_action": null, "control_profile": null, "host_model": null, "remote_actuation": null },
  "acceptance": { "counter_reconciliation": null, "minimum_critical_on_time_pdr": null, "negative_case": null },
  "evidence": { "required_artifacts": null, "operator_notes_required": null, "topology_photo_required": null },
  "_todo": [
    {
      "path": "/setup",
      "action": "Bind this held-out condition to a completed stable-baseline block before changing workload shape.",
      "method": "Copy the validated physical board identities, channel, placement, and firmware reference from one archived baseline block. If any of those facts change, declare a new calibration block instead of calling the traces held out.",
      "done_when": "The ready file cites a specific baseline evidence bundle and no network or firmware change is hidden inside the prediction condition."
    },
    {
      "path": "/traffic",
      "action": "Design one recoverable bulk load step while preserving the frozen critical stream.",
      "method": "Use pilots outside the final set to increase only bulk period, payload, or burst until critical behavior changes measurably but the queue and radio recover. Place the step after warm-up, keep it inside measurement, and define its target and duration before observing any held-out result.",
      "done_when": "The ready file fixes all streams, event time, duration, seed, and repetition count, and pilot logs show a visible but recoverable disturbance."
    },
    {
      "path": "/treatment",
      "action": "Keep this experiment as a shadow-model prediction test, not a disguised controller test.",
      "method": "Set prediction mode, enable the host model, disable remote actuation, choose a critical service floor from the baseline, and require raw traces plus a prediction report that identifies calibration data, model revision, prediction horizon, all three models, calibrated-region status, and observation-integrity status.",
      "done_when": "No held-out observation is used to tune model parameters or gate thresholds and the report scores all models on exactly the same horizons."
    }
  ]
}


Promotion sequence:

1. JSONC receives only pilot-backed/frozen decisions.
2. Every machine-readable value is copied into strict JSON without comments.
3. All three `_todo` objects remain until setup binding, step semantics, treatment, and evidence are complete.
4. Strict file is validated against schema and deterministic cross-field validator.
5. Parser conversion proves every string fits fixed C storage and every enum maps exactly.
6. `state` changes to `ready` and `_todo` becomes empty in the same reviewed change.
7. Original ready bytes and digest are archived before model/broker connection.
8. A later edit creates a new manifest digest/run; old evidence is not overwritten.

No example number in this notebook is copied into the strict file unless the corresponding pilot/evidence actually produced it.

### Machine-facing contract: `schemas/experiment.schema.json`

In [ ]:
%%writefile /content/cldt_scratch/experiment.schema.json
{
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "$id": "https://github.com/Reinathanajah/CLDT-Thread/blob/main/schemas/experiment.schema.json",
  "title": "CLDT Experiment Manifest",
  "description": "A deliberately small, two-stage contract. A template may retain null decisions and actionable _todo entries; a ready manifest may not.",
  "type": "object",
  "additionalProperties": false,
  "required": ["schema_version", "state", "experiment_id", "title", "purpose", "_todo"],
  "properties": {
    "schema_version": { "const": "2.0" },
    "state": { "enum": ["template", "ready"] },
    "experiment_id": { "$ref": "#/$defs/identifier" },
    "title": { "type": "string", "minLength": 8, "maxLength": 120 },
    "purpose": { "$ref": "#/$defs/purpose" },
    "_todo": {
      "type": "array",
      "uniqueItems": true,
      "items": { "$ref": "#/$defs/todo" }
    },
    "setup": { "$ref": "#/$defs/template_setup" },
    "execution": { "$ref": "#/$defs/template_execution" },
    "traffic": { "$ref": "#/$defs/template_traffic" },
    "scenario": { "$ref": "#/$defs/template_scenario" },
    "treatment": { "$ref": "#/$defs/template_treatment" },
    "acceptance": { "$ref": "#/$defs/template_acceptance" },
    "evidence": { "$ref": "#/$defs/template_evidence" }
  },
  "allOf": [
    {
      "if": { "properties": { "state": { "const": "template" } } },
      "then": { "properties": { "_todo": { "minItems": 1 } } }
    },
    {
      "if": { "properties": { "state": { "const": "ready" } } },
      "then": {
        "required": ["setup", "execution", "traffic", "scenario", "treatment", "acceptance", "evidence"],
        "properties": {
          "_todo": { "maxItems": 0 },
          "setup": { "$ref": "#/$defs/ready_setup" },
          "execution": { "$ref": "#/$defs/ready_execution" },
          "traffic": { "$ref": "#/$defs/ready_traffic" },
          "scenario": { "$ref": "#/$defs/ready_scenario" },
          "treatment": { "$ref": "#/$defs/ready_treatment" },
          "acceptance": { "$ref": "#/$defs/ready_acceptance" },
          "evidence": { "$ref": "#/$defs/ready_evidence" }
        }
      }
    }
  ],
  "$defs": {
    "identifier": {
      "type": "string",
      "minLength": 3,
      "maxLength": 64,
      "pattern": "^[a-z0-9][a-z0-9._-]*$"
    },
    "non_empty_text": {
      "type": "string",
      "minLength": 3,
      "maxLength": 512
    },
    "purpose": {
      "type": "object",
      "additionalProperties": false,
      "required": ["question", "comparison", "primary_metric"],
      "properties": {
        "question": { "$ref": "#/$defs/non_empty_text" },
        "comparison": { "$ref": "#/$defs/non_empty_text" },
        "primary_metric": { "$ref": "#/$defs/non_empty_text" }
      }
    },
    "todo": {
      "type": "object",
      "additionalProperties": false,
      "required": ["path", "action", "method", "done_when"],
      "properties": {
        "path": { "type": "string", "pattern": "^/" },
        "action": { "$ref": "#/$defs/non_empty_text" },
        "method": { "$ref": "#/$defs/non_empty_text" },
        "done_when": { "$ref": "#/$defs/non_empty_text" }
      }
    },
    "node": {
      "type": "object",
      "additionalProperties": false,
      "required": ["id", "board", "role"],
      "properties": {
        "id": { "$ref": "#/$defs/identifier" },
        "board": { "enum": ["esp32-s3", "esp32-c6"] },
        "role": { "enum": ["gateway", "radio_coprocessor", "router_endpoint", "low_power_endpoint"] }
      }
    },
    "template_setup": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "nodes": { "type": ["array", "null"], "maxItems": 4, "uniqueItems": true, "items": { "$ref": "#/$defs/node" } },
        "thread_channel": { "type": ["integer", "null"], "minimum": 11, "maximum": 26 },
        "placement": { "type": ["string", "null"], "maxLength": 160 },
        "firmware_reference": { "type": ["string", "null"], "maxLength": 160 }
      }
    },
    "ready_setup": {
      "type": "object",
      "additionalProperties": false,
      "required": ["nodes", "thread_channel", "placement", "firmware_reference"],
      "properties": {
        "nodes": { "type": "array", "minItems": 4, "maxItems": 4, "uniqueItems": true, "items": { "$ref": "#/$defs/node" } },
        "thread_channel": { "type": "integer", "minimum": 11, "maximum": 26 },
        "placement": { "$ref": "#/$defs/non_empty_text" },
        "firmware_reference": { "$ref": "#/$defs/non_empty_text" }
      }
    },
    "template_execution": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "warmup_s": { "type": ["integer", "null"], "minimum": 5, "maximum": 300 },
        "measurement_s": { "type": ["integer", "null"], "minimum": 30, "maximum": 3600 },
        "cooldown_s": { "type": ["integer", "null"], "minimum": 5, "maximum": 300 },
        "repetitions": { "type": ["integer", "null"], "minimum": 1, "maximum": 30 },
        "seed": { "type": ["integer", "null"], "minimum": 0, "maximum": 4294967295 }
      }
    },
    "ready_execution": {
      "type": "object",
      "additionalProperties": false,
      "required": ["warmup_s", "measurement_s", "cooldown_s", "repetitions", "seed"],
      "properties": {
        "warmup_s": { "type": "integer", "minimum": 5, "maximum": 300 },
        "measurement_s": { "type": "integer", "minimum": 30, "maximum": 3600 },
        "cooldown_s": { "type": "integer", "minimum": 5, "maximum": 300 },
        "repetitions": { "type": "integer", "minimum": 1, "maximum": 30 },
        "seed": { "type": "integer", "minimum": 0, "maximum": 4294967295 }
      }
    },
    "stream": {
      "type": "object",
      "additionalProperties": false,
      "required": ["id", "source", "class", "period_ms", "payload_bytes", "deadline_ms", "burst_packets"],
      "properties": {
        "id": { "$ref": "#/$defs/identifier" },
        "source": { "$ref": "#/$defs/identifier" },
        "class": { "enum": ["control", "critical", "telemetry", "bulk"] },
        "period_ms": { "type": ["integer", "null"], "minimum": 10, "maximum": 3600000 },
        "payload_bytes": { "type": ["integer", "null"], "minimum": 1, "maximum": 256 },
        "deadline_ms": { "type": ["integer", "null"], "minimum": 10, "maximum": 3600000 },
        "burst_packets": { "type": ["integer", "null"], "minimum": 1, "maximum": 100 }
      }
    },
    "ready_stream": {
      "type": "object",
      "additionalProperties": false,
      "required": ["id", "source", "class", "period_ms", "payload_bytes", "deadline_ms", "burst_packets"],
      "properties": {
        "id": { "$ref": "#/$defs/identifier" },
        "source": { "$ref": "#/$defs/identifier" },
        "class": { "enum": ["control", "critical", "telemetry", "bulk"] },
        "period_ms": { "type": "integer", "minimum": 10, "maximum": 3600000 },
        "payload_bytes": { "type": "integer", "minimum": 1, "maximum": 256 },
        "deadline_ms": { "type": "integer", "minimum": 10, "maximum": 3600000 },
        "burst_packets": { "type": "integer", "minimum": 1, "maximum": 100 }
      }
    },
    "template_traffic": {
      "type": "object",
      "additionalProperties": false,
      "properties": { "streams": { "type": ["array", "null"], "maxItems": 4, "uniqueItems": true, "items": { "$ref": "#/$defs/stream" } } }
    },
    "ready_traffic": {
      "type": "object",
      "additionalProperties": false,
      "required": ["streams"],
      "properties": { "streams": { "type": "array", "minItems": 1, "maxItems": 4, "uniqueItems": true, "items": { "$ref": "#/$defs/ready_stream" } } }
    },
    "template_scenario": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "event": { "enum": ["none", "load_step", "observation_pause", "endpoint_restart", "topology_shift", null] },
        "at_s": { "type": ["integer", "null"], "minimum": 0, "maximum": 3600 },
        "duration_s": { "type": ["integer", "null"], "minimum": 0, "maximum": 3600 },
        "target": { "type": ["string", "null"], "maxLength": 120 }
      }
    },
    "ready_scenario": {
      "type": "object",
      "additionalProperties": false,
      "required": ["event", "at_s", "duration_s", "target"],
      "properties": {
        "event": { "enum": ["none", "load_step", "observation_pause", "endpoint_restart", "topology_shift"] },
        "at_s": { "type": "integer", "minimum": 0, "maximum": 3600 },
        "duration_s": { "type": "integer", "minimum": 0, "maximum": 3600 },
        "target": { "$ref": "#/$defs/non_empty_text" }
      }
    },
    "template_treatment": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "mode": { "enum": ["baseline", "prediction", "gated_control", "safety", "smp", "power", null] },
        "candidate_action": { "enum": ["none", "bulk_rate_reduce", "phase_stagger", "power_profile", null] },
        "control_profile": { "type": ["string", "null"], "maxLength": 160 },
        "host_model": { "type": ["boolean", "null"] },
        "remote_actuation": { "type": ["boolean", "null"] }
      }
    },
    "ready_treatment": {
      "type": "object",
      "additionalProperties": false,
      "required": ["mode", "candidate_action", "control_profile", "host_model", "remote_actuation"],
      "properties": {
        "mode": { "enum": ["baseline", "prediction", "gated_control", "safety", "smp", "power"] },
        "candidate_action": { "enum": ["none", "bulk_rate_reduce", "phase_stagger", "power_profile"] },
        "control_profile": { "$ref": "#/$defs/non_empty_text" },
        "host_model": { "type": "boolean" },
        "remote_actuation": { "type": "boolean" }
      }
    },
    "template_acceptance": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "counter_reconciliation": { "type": ["boolean", "null"] },
        "minimum_critical_on_time_pdr": { "type": ["number", "null"], "minimum": 0, "maximum": 1 },
        "negative_case": { "type": ["string", "null"], "maxLength": 120 }
      }
    },
    "ready_acceptance": {
      "type": "object",
      "additionalProperties": false,
      "required": ["counter_reconciliation", "minimum_critical_on_time_pdr", "negative_case"],
      "properties": {
        "counter_reconciliation": { "const": true },
        "minimum_critical_on_time_pdr": { "type": "number", "minimum": 0, "maximum": 1 },
        "negative_case": { "$ref": "#/$defs/non_empty_text" }
      }
    },
    "template_evidence": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "required_artifacts": { "type": ["array", "null"], "maxItems": 8, "uniqueItems": true, "items": { "$ref": "#/$defs/non_empty_text" } },
        "operator_notes_required": { "type": ["boolean", "null"] },
        "topology_photo_required": { "type": ["boolean", "null"] }
      }
    },
    "ready_evidence": {
      "type": "object",
      "additionalProperties": false,
      "required": ["required_artifacts", "operator_notes_required", "topology_photo_required"],
      "properties": {
        "required_artifacts": { "type": "array", "minItems": 4, "maxItems": 8, "uniqueItems": true, "items": { "$ref": "#/$defs/non_empty_text" } },
        "operator_notes_required": { "const": true },
        "topology_photo_required": { "type": "boolean" }
      }
    }
  }
}


Schema review relevant to Phase 4:

1. Ready `setup.nodes` requires exactly four objects and channel 11–26. Karena `uniqueItems` hanya menolak object yang identik, cross-field admission juga membuktikan empat ID unik dan tepat satu object untuk setiap physical role.
2. Streams are bounded to one–four; period/deadline 10–3,600,000 ms, payload 1–256 bytes, and burst 1–100.
3. Ready scenario requires event, time, duration, and non-empty target.
4. Ready treatment requires all five fields, but schema alone does not enforce the prediction/no-action/model-on/remote-off combination.
5. Ready acceptance fixes `counter_reconciliation` to true.
6. Ready evidence requires 4–8 unique artifact strings.
7. `additionalProperties: false` means baseline binding, matrices, feature revision, horizon cadence, and before/during step values cannot be appended ad hoc.
8. Ready text may be 512 chars while several runtime buffers are 161 bytes; parser admission remains stricter.
9. Schema permits one repetition, but meaningful run-aware inference may require more; final count comes from pilot variance/time budget, not schema minimum.
10. Syntax/schema PASS does not establish experimental eligibility. Cross-field, physical identity, calibration freeze, and evidence gates remain separate.

## Step 12: Calibration, Pilot, dan Freeze
### Tiga Dataset Roles yang Tidak Dicampur

| Data role | Dipakai untuk | Tidak dipakai untuk |
|---|---|---|
| Engineering pilot | menemukan disturbance bulk yang visible tetapi recoverable; memilih measurement duration dan feasible bounds | final model fit, gate threshold, atau held-out claim |
| Calibration block | fit model parameters, noise matrices, naive window, normalization, interval construction, and support description | final held-out score |
| Held-out block | score all frozen variants on predeclared load step | feature selection, parameter tuning, interval tuning, threshold selection |

Freeze record berisi human labels berikut; label bukan variable code baru:

| Freeze item | Evidence yang dicatat |
|---|---|
| Calibration run identities | exact run directory IDs/digests |
| Held-out run plan | planned count, seed plan, distinct boot rule, invalidation rule |
| Model code/source | source commit and build/tool versions |
| Frozen model artifact | exact calibration/matrix/feature/code identity and digest assigned before held-out access |
| Runtime `model_revision` semantics | state-generation counter; may advance only on accepted logical state changes and is captured per prediction, never used as the immutable artifact ID |
| State order | five-state mapping from `DESIGN.md` |
| `F`, `H`, `Q`, `R`, initial covariance | exact matrix artifact + digest |
| Naive window | exact historical duration/count |
| Feature allowlists | network and cross labels with exact source fields |
| Fit/loss | same family/loss for network and cross |
| Horizon schedule | issue cadence, start offset, duration, boundary semantics |
| Per-horizon relative-error denominator rule | explicit near-zero observed-target handling |
| Improvement-ratio denominator guard | frozen positive tolerance for network P95 error; non-evaluable case predeclared |
| Prediction interval method | exact bound metric, units, construction, and calibration-only inputs |
| Random seeds | fitting/bootstrap seeds and derivation |
| Primary decision | $\mathrm{LCB}_{0.95}(\Delta) > 0.15$ |
| Service floor | unchanged critical on-time delivery floor |

Freeze happens before final held-out outcome is opened. Calibration baseline dan held-out block memakai exact load-step-capable firmware/configuration yang sama; hanya admitted workload schedule yang berbeda. Any edit to frozen items creates a new artifact identity and a new held-out plan; scores from different frozen artifacts are not merged. Runtime `model_revision` remains a separately recorded state-generation counter.

## Step 13: Physical Held-Out Runs
### Empat Board yang Sama, Satu Perubahan Workload

Tidak ada pembelian atau sensor baru. Meja tetap ESP32-S3 gateway, dedicated ESP32-C6 RCP, endpoint A, endpoint B, powered hub, verified data cables, private AP, host, dan optional logic analyzer yang sudah ada.

Urutan satu reportable run:

1. Ready manifest bytes and digest are verified; control profile/calibration identity and frozen model artifacts resolve.
2. New nonzero ledger `run_id`, coordinator boot identity, and non-overwriting run directory are created before broker connection.
3. Board labels, binary hashes, channel, placement, UART, power, actual Thread role/parent/partition, clock health, and zero/baseline counters are checked against the frozen block.
4. Warm-up is recorded but excluded from primary horizon target.
5. Measurement boundary is stored in host monotonic time. Raw observation enters recorder before decode/model update.
6. Three frozen predictions are issued on exact same future horizons. No held-out observation updates model parameters.
7. Only the declared bulk load step occurs at the declared release boundary. Effective time, target, before/during values, and local acknowledgement are recorded.
8. Critical stream, topology, firmware, channel, placement, and physical power path do not change.
9. Completed horizons are reconciled and scored once; missing/integrity-failed horizons remain visible.
10. Step restoration and recovery occur before cooldown according to the frozen scenario.
11. Cooldown stops new release, accounts/drains bounded ownership, captures final device counters, and resolves pending horizon status.
12. Item audit and aggregate reconciliation run. A mismatch makes the run invalid for performance interpretation.
13. Exactly one terminal status—complete, invalid, or interrupted—is written with operator notes.
14. Every planned run, including failed/invalid/interrupted runs, remains in the experiment ledger.

Independent physical runs with distinct boot cycles are the primary experimental units. Windows from one long run do not become independent replications. Unexpected node movement, role/parent/partition change, second disturbance, wrong image/profile, missed step, incomplete evidence, or reconciliation failure triggers the predeclared invalid rule; raw evidence is preserved.

## Step 14: Frozen Scoring dan Interpretation
### Primary Pairing, Run-Aware Uncertainty, dan Tiga Outcome

For each eligible horizon (h), point error uses one frozen target and near-zero rule. The project then computes held-out P95 error for each model. Improvement of cross-layer over network-only is:

$$
\Delta =
\frac{
\operatorname{Error}(M_{\text{network}})
-
\operatorname{Error}(M_{\text{cross}})
}{
\operatorname{Error}(M_{\text{network}})
}
$$

The ratio is evaluated only when the network-model P95 error is greater than a small positive tolerance frozen from metric resolution and numerical precision before held-out access. If the denominator is zero or within that tolerance, `Delta` is marked non-evaluable, both absolute errors and their paired difference are still reported, and the relative 15% criterion cannot receive a positive verdict. It is not converted into infinity or silently stabilized after seeing the result.

The predeclared decision, when the ratio is evaluable, is:

$$
\operatorname{LCB}_{0.95}(\Delta) > 0.15
$$

Interpretation is limited to three outcomes:

1. Lower confidence bound above 0.15: evidence supports improvement beyond the 15% engineering threshold under this exact topology/environment/revision.
2. Point estimate above 0.15 but lower bound not above 0.15: observed improvement is promising but not established at the required confidence.
3. Point estimate at or below 0.15: observed improvement does not reach the threshold.

Failing to reject the null is not proof of equivalence. A negative result is frozen and reported; network-only is not promoted as replacement actuator. Remote actuation stays disabled unless later Week-5 work is separately eligible.

| Frozen output | Minimum content |
|---|---|
| Per-horizon prediction table | run, variant, revision, issue time, exact horizon, point estimate, interval, source/integrity/support status |
| Per-horizon observed table | exact horizon identity, reconciled critical outcome, completeness |
| Pairing audit | three variants present on every compared horizon; no nearest-time join |
| Primary model table CSV | naive/network/cross P95 error and run-aware uncertainty |
| $\hat{\Delta}$, denominator status, and one-sided 95% LCB | whole-run resampling or run-level analysis; explicit non-evaluable reason when network error is near zero |
| Interval coverage | numerator, denominator, method, and integrity exclusions |
| Run ledger | every complete/invalid/interrupted run and reason |
| Service result | critical on-time delivery and deadline miss beside prediction result |
| Freeze artifact | model/matrix/feature/horizon/calibration identities and digests |
| Reproduction record | exact environment/dependencies and nonzero failure behavior |

Feature-group ablation and context shift are not used to rescue the primary result. They remain conditional after the frozen three-model comparison and Week-5 safety closure.

## Phase 4 Delivery Checklist
### Urutan Implementasi dan Kriteria Penyelesaian

| No. | Pekerjaan | Selesai ketika |
|---:|---|---|
| 1 | bawa evidence gate Phase 3 | repeated baseline raw lifecycle dan aggregate reconciliation lulus |
| 2 | tutup observation payload/time mapping | cross-layer source bytes dapat di-decode dengan fixed contract |
| 3 | tutup node/order/calibration/horizon identities | duplicate, stale, cross-run, and held-out update dapat ditolak |
| 4 | tentukan Kalman ownership and numerical contract | matrices/state order/init/missing/singular rules dapat diuji |
| 5 | bekukan naive/network/cross feature and parity contract | only feature visibility differs for matched candidate models |
| 6 | tutup bounded pending observation/prediction ownership in coordinator | callback cepat, horizon tidak hilang, stop tidak starve |
| 7 | tutup executable local load-step semantics | before/during/restoration/acknowledgement machine-readable dan remote-off |
| 8 | sediakan deterministic model verification owner | numerical, ordering, leakage, freeze, and parity cases pass |
| 9 | selesaikan Week-4 subset `reproduce.py` | raw preflight → paired score → run-aware uncertainty → frozen CSV |
| 10 | jalankan engineering pilots | disturbance recoverable dan final plan dipilih tanpa memakai final outcome |
| 11 | fit calibration-only models | matrices/window/normalization/interval/support artifacts tersedia |
| 12 | freeze model and experiment plan | artifact digest, feature, horizon, seed, both denominator rules, interval target/units, and service floor immutable |
| 13 | promote load-step manifest only if eligible | ready JSON no null/TODO, schema/cross-field/buffer validation pass |
| 14 | run every planned held-out physical repetition | one scenario, raw-first, no parameter update, no remote actuation |
| 15 | score exact paired horizons | all variants, integrity failures, invalid/interrupted runs retained |
| 16 | freeze positive/inconclusive/negative result | report and evidence bundle replay without hidden selection |

### Closure Week 4

- [ ] Technical source and notebook revision identities recorded.
- [ ] Every Phase-3 input bundle used for calibration passes item and aggregate reconciliation.
- [ ] Observation payload, source-presence, clock-domain, and feature mapping are fixed and tested.
- [ ] Node mapping and duplicate/out-of-order behavior are bounded.
- [ ] Calibration transition prevents any held-out fitting.
- [ ] Network/cross share model family, loss, calibration block, horizon, and fit procedure.
- [ ] Naive historical window is frozen.
- [ ] Kalman matrices, initial state/covariance, and numerical failure rules are frozen.
- [ ] Three predictions share exact issue/start/end boundaries.
- [ ] Prediction/outcome join uses identity, not row order or nearest timestamp.
- [ ] Load step is local, predeclared, acknowledged, recoverable, and changes only bulk workload.
- [ ] `candidate_action` is none and remote actuation remains false.
- [ ] Every physical run has raw-first evidence and one terminal status.
- [ ] Missing/stale/unreconciled horizons stay visible.
- [ ] Independent runs—not windows—drive primary uncertainty.
- [ ] $\hat{\Delta}$ denominator guard/non-evaluable status, one-sided LCB when defined, metric-bound interval coverage, service result, and limitation are reported.
- [ ] Model/test/dependency gaps are closed by real repo owners; no fictional path or PASS remains.
- [ ] Fidelity gate, policy, command, safety cases, ablation, topology shift, SMP, power, and dashboard remain untouched.

> **Exit rule:** tanpa frozen Week-4 shadow result, tidak ada physical-context-depth claim dan tidak ada remote actuation. Frozen negative result tetap merupakan penyelesaian yang valid; incomplete identity, leakage, atau unreconciled evidence bukan.